# 7 · Classification — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch07_classification/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch07_classification/03_notebook.ipynb`, créée par `python tools/start_chapter.py 7`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 7.Q1 à 7.Q11, 7.R1 à 7.R3, 7.1 à 7.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 7.11 | Des œufs en 2D : données, régions et frontière de décision | 📦 | ★ | 15 |
|  | 7.12 | k-means sur deux lunes : où tombera la coupure ? | 🔮 | ★ | 10 |
|  | 7.13 | Distances au carré vectorisées : pairwise_sq_distances | 🔨 | ★★ | 20 |
|  | 7.14 | Le classifieur du centroïde le plus proche | 🔨 | ★★ | 30 |
|  | 7.15 | Carte de probabilité et politique de seuil pour les œufs | 📈 | ★★ | 25 |
|  | 7.16 | Un-contre-tous avec des centroïdes : quelle classe sera sacrifiée ? | 🔮 | ★★ | 15 |
| B | 7.17 | Manchots sans labels : k-means face aux espèces | 📦 | ★★ | 25 |
|  | 7.18 | Formes arbitraires et bruit : DBSCAN et HDBSCAN | 📦 | ★★ | 25 |
|  | 7.19 | Distance au plus proche voisin quand la dimension grimpe | 🔮 | ★★ | 15 |
|  | 7.20 | Densité, plus proche voisin et concentration des distances | 🔬 | ★★ | 30 |
|  | 7.21 | Reproduire les figures 7.27 et 7.30 (boule/cube, hyper-orange) | 🎨 | ★★ | 25 |
| C | 7.22 | Un-contre-tous générique : OneVsRestClassifier | 🔨 | ★★★ | 40 |
|  | 7.23 | Un-contre-un générique : OneVsOneClassifier | 🔨 | ★★★ | 45 |
|  | 7.24 | OvR, OvO ou multi-classe natif : accuracy, nombre de modèles, temps | 🔬 | ★★★ | 35 |
| D | 7.25 | Initialisation k-means++ | 🔨 | ★★★ | 35 |
|  | 7.26 | k-means de Lloyd : la classe KMeans | 🔨 | ★★★ | 60 |
|  | 7.27 | k-means piégé : quatre bugs à débusquer | 🐛 | ★★★ | 30 |
|  | 7.28 | Coefficient de silhouette | 🔨 | ★★★ | 40 |
|  | 7.29 | Choisir k : coude de l'inertie et silhouette, de k = 2 à 7 | 🔬 | ★★★ | 35 |
|  | 7.30 | Phénomène de Hughes : des features de bruit qui font chuter l'accuracy | 🔬 | ★★★ | 40 |
|  | 7.31 | Défi : retrouver les espèces de manchots sans labels | 🏆 | ★★★ | 60 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")  # your own library

## Objectifs et rappel express

- Lire des régions et des frontières de décision, et régler un seuil d'après le coût des erreurs.
- Traiter plusieurs classes avec des classifieurs binaires : un-contre-tous et un-contre-un.
- Programmer le centroïde le plus proche et k-means (Lloyd, k-means++), et choisir $k$ avec l'inertie et la silhouette ; savoir quand préférer DBSCAN ou HDBSCAN.
- Mesurer ce que la dimension fait aux densités et aux distances.

**Rappel express.** $N_{\text{OvR}} = K$, $N_{\text{OvO}} = \frac{K(K-1)}{2}$ ; seuil de coût minimal $t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$ ; $\lVert \mathbf{a} - \mathbf{b} \rVert^2 = \lVert \mathbf{a} \rVert^2 - 2\,\mathbf{a} \cdot \mathbf{b} + \lVert \mathbf{b} \rVert^2$ ; inertie $J = \sum_i \lVert \mathbf{x}_i - \boldsymbol{\mu}_{c_i} \rVert^2$ ; silhouette $s = \frac{b - a}{\max(a, b)}$ ; densité $\rho = \frac{n}{b^d}$ ; peau d'une boule $1 - (1 - \varepsilon)^d$. En Python : `sklearn.cluster.KMeans`, `DBSCAN`, `HDBSCAN`, `sklearn.metrics.silhouette_score`, `sklearn.multiclass`, `rng.choice(n, p=w / w.sum())`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 7.1 à 7.4, ∂ 7.5 et 7.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.375` (en Python, le séparateur décimal est un **point** ; `0,375` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"D"`) ; plusieurs nombres : une liste (`[2, 5]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », le quiz Q10, ∂ 7.7, la réflexion et l'entretien se corrigent avec `05_solutions.md`.

**7.Q1 — Label, prédiction, vérité terrain : le vocabulaire**

In [ ]:
# 7.Q1: the values you found on paper
answer_7_Q1a = ...  # a) the letter of the expression that is NOT the expert's class, e.g. "A"
answer_7_Q1b = ...  # b) True or False
answer_7_Q1c = ...  # c) True or False

for letter, answer in zip("abc", [answer_7_Q1a, answer_7_Q1b, answer_7_Q1c]):
    wb.check(f"7.Q1{letter}", answer)

**7.Q2 — Binaire, multi-classe ou multi-étiquette ? Cinq situations**

In [ ]:
# 7.Q2: the values you found on paper
answer_7_Q2a = ...  # a) "B", "C" or "E"
answer_7_Q2b = ...  # b) "B", "C" or "E"
answer_7_Q2c = ...  # c) "B", "C" or "E"
answer_7_Q2d = ...  # d) "B", "C" or "E"
answer_7_Q2e = ...  # e) "B", "C" or "E"

for letter, answer in zip("abcde", [answer_7_Q2a, answer_7_Q2b, answer_7_Q2c, answer_7_Q2d, answer_7_Q2e]):
    wb.check(f"7.Q2{letter}", answer)

**7.Q3 — Régions et frontières de décision**

In [ ]:
# 7.Q3: the values you found on paper
answer_7_Q3a = ...  # a) True or False
answer_7_Q3b = ...  # b) a whole number
answer_7_Q3c = ...  # c) True or False
answer_7_Q3d = ...  # d) True or False
answer_7_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q3a, answer_7_Q3b, answer_7_Q3c, answer_7_Q3d, answer_7_Q3e]):
    wb.check(f"7.Q3{letter}", answer)

**7.Q4 — Classes qui se recouvrent : probabilités et politique de seuil**

In [ ]:
# 7.Q4: the values you found on paper
answer_7_Q4a = ...  # a) True or False
answer_7_Q4b = ...  # b) True or False
answer_7_Q4c = ...  # c) "A", "B" or "C"
answer_7_Q4d = ...  # d) the threshold, 3 decimals

for letter, answer in zip("abcd", [answer_7_Q4a, answer_7_Q4b, answer_7_Q4c, answer_7_Q4d]):
    wb.check(f"7.Q4{letter}", answer)

**7.Q5 — Cinq mesures par œuf : ce que change (ou non) la dimension**

In [ ]:
# 7.Q5: the values you found on paper
answer_7_Q5a = ...  # a) a whole number
answer_7_Q5b = ...  # b) True or False
answer_7_Q5c = ...  # c) True or False
answer_7_Q5d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_7_Q5a, answer_7_Q5b, answer_7_Q5c, answer_7_Q5d]):
    wb.check(f"7.Q5{letter}", answer)

**7.Q6 — Un-contre-tous : combien de modèles, quelle décision ?**

In [ ]:
# 7.Q6: the values you found on paper
answer_7_Q6a = ...  # a) a whole number
answer_7_Q6b = ...  # b) a letter
answer_7_Q6c = ...  # c) True or False
answer_7_Q6d = ...  # d) a letter

for letter, answer in zip("abcd", [answer_7_Q6a, answer_7_Q6b, answer_7_Q6c, answer_7_Q6d]):
    wb.check(f"7.Q6{letter}", answer)

**7.Q7 — Un-contre-un : duels, votes et coût**

In [ ]:
# 7.Q7: the values you found on paper
answer_7_Q7a = ...  # a) a whole number
answer_7_Q7b = ...  # b) True or False
answer_7_Q7c = ...  # c) True or False
answer_7_Q7d = ...  # d) a letter
answer_7_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q7a, answer_7_Q7b, answer_7_Q7c, answer_7_Q7d, answer_7_Q7e]):
    wb.check(f"7.Q7{letter}", answer)

**7.Q8 — Ce que k-means ne peut pas deviner tout seul**

In [ ]:
# 7.Q8: the values you found on paper
answer_7_Q8a = ...  # a) True or False
answer_7_Q8b = ...  # b) a letter
answer_7_Q8c = ...  # c) True or False
answer_7_Q8d = ...  # d) True or False
answer_7_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q8a, answer_7_Q8b, answer_7_Q8c, answer_7_Q8d, answer_7_Q8e]):
    wb.check(f"7.Q8{letter}", answer)

**7.Q9 — Densité d'échantillons quand les features s'accumulent**

In [ ]:
# 7.Q9: the values you found on paper
answer_7_Q9a = ...  # a) a whole number
answer_7_Q9b = ...  # b) 2 decimals
answer_7_Q9c = ...  # c) 4 decimals
answer_7_Q9d = ...  # d) True or False
answer_7_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_7_Q9a, answer_7_Q9b, answer_7_Q9c, answer_7_Q9d, answer_7_Q9e]):
    wb.check(f"7.Q9{letter}", answer)

**7.Q11 — Géométrie déroutante en grande dimension**

In [ ]:
# 7.Q11: the values you found on paper
answer_7_Q11a = ...  # a) True or False
answer_7_Q11b = ...  # b) True or False
answer_7_Q11c = ...  # c) True or False
answer_7_Q11d = ...  # d) a letter

for letter, answer in zip("abcd", [answer_7_Q11a, answer_7_Q11b, answer_7_Q11c, answer_7_Q11d]):
    wb.check(f"7.Q11{letter}", answer)

**7.R1 — Ch. 6 : entropie d'un cluster pur et d'un cluster mélangé**

In [ ]:
# 7.R1: the values you found on paper
answer_7_R1a = ...  # a) in bits
answer_7_R1b = ...  # b) in bits
answer_7_R1c = ...  # c) in bits
answer_7_R1d = ...  # d) in bits, 3 decimals
answer_7_R1e = ...  # e) in bits, 3 decimals

for letter, answer in zip("abcde", [answer_7_R1a, answer_7_R1b, answer_7_R1c, answer_7_R1d, answer_7_R1e]):
    wb.check(f"7.R1{letter}", answer)

**7.R2 — Ch. 4 : probabilité a posteriori « fécondé » par la règle de Bayes**

In [ ]:
# 7.R2: the values you found on paper
answer_7_R2a = ...  # a) 3 decimals
answer_7_R2b = ...  # b) True or False
answer_7_R2c = ...  # c) 3 decimals
answer_7_R2d = ...  # d) 3 decimals

for letter, answer in zip("abcd", [answer_7_R2a, answer_7_R2b, answer_7_R2c, answer_7_R2d]):
    wb.check(f"7.R2{letter}", answer)

**7.R3 — 0B : développer ‖a − b‖² avec le produit scalaire**

In [ ]:
# 7.R3: the values you found on paper
answer_7_R3a = ...  # a) a whole number
answer_7_R3b = ...  # b) a whole number
answer_7_R3c = ...  # c) a whole number
answer_7_R3d = ...  # d) a whole number

for letter, answer in zip("abcd", [answer_7_R3a, answer_7_R3b, answer_7_R3c, answer_7_R3d]):
    wb.check(f"7.R3{letter}", answer)

**Ex 7.1 — Compter les classifieurs OvR et OvO**

In [ ]:
# Ex 7.1: the values you found on paper
answer_7_1a = ...  # a) [N_OvR, N_OvO] for K = 3
answer_7_1b = ...  # b) [N_OvR, N_OvO] for K = 10
answer_7_1c = ...  # c) [N_OvR, N_OvO] for K = 26
answer_7_1d = ...  # d) [N_OvR, N_OvO] for K = 100
answer_7_1e = ...  # e) the smallest K
answer_7_1f = ...  # f) total number of training examples, one-versus-rest
answer_7_1g = ...  # g) total number of training examples, one-versus-one
answer_7_1h = ...  # h) how many times cheaper, 1 decimal

for letter, answer in zip("abcdefgh", [answer_7_1a, answer_7_1b, answer_7_1c, answer_7_1d, answer_7_1e, answer_7_1f, answer_7_1g, answer_7_1h]):
    wb.check(f"7.1{letter}", answer)

**Ex 7.2 — Dépouiller les votes d'un un-contre-un à quatre classes**

In [ ]:
# Ex 7.2: the values you found on paper
answer_7_2a = ...  # a) votes [A, B, C, D], first point
answer_7_2b = ...  # b) a letter
answer_7_2c = ...  # c) a whole number
answer_7_2d = ...  # d) votes [A, B, C, D], second point
answer_7_2e = ...  # e) a letter
answer_7_2f = ...  # f) a letter
answer_7_2g = ...  # g) True or False
answer_7_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_7_2a, answer_7_2b, answer_7_2c, answer_7_2d, answer_7_2e, answer_7_2f, answer_7_2g, answer_7_2h]):
    wb.check(f"7.2{letter}", answer)

**Ex 7.3 — Une itération de k-means à la main**

In [ ]:
# Ex 7.3: the values you found on paper
answer_7_3a = ...  # a) cluster (1 or 2) of P1 to P6
answer_7_3b = ...  # b) the inertia with the starting centres
answer_7_3c = ...  # c) the two new centres [[x1, y1], [x2, y2]], 2 decimals
answer_7_3d = ...  # d) 2 decimals
answer_7_3e = ...  # e) the number of the point (1 to 6)
answer_7_3f = ...  # f) the centres after the second update, 3 decimals
answer_7_3g = ...  # g) the final inertia, 2 decimals
answer_7_3h = ...  # h) True or False
answer_7_3i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_7_3a, answer_7_3b, answer_7_3c, answer_7_3d, answer_7_3e, answer_7_3f, answer_7_3g, answer_7_3h, answer_7_3i]):
    wb.check(f"7.3{letter}", answer)

**Ex 7.4 — Densité d'échantillons et nombre d'œufs nécessaires**

In [ ]:
# Ex 7.4: the values you found on paper
answer_7_4a = ...  # a) [d = 1, d = 2, d = 3, d = 4], 3 decimals
answer_7_4b = ...  # b) a whole number
answer_7_4c = ...  # c) a whole number
answer_7_4d = ...  # d) a whole number
answer_7_4e = ...  # e) the smallest dimension
answer_7_4f = ...  # f) 3 decimals
answer_7_4g = ...  # g) 2 decimals

for letter, answer in zip("abcdefg", [answer_7_4a, answer_7_4b, answer_7_4c, answer_7_4d, answer_7_4e, answer_7_4f, answer_7_4g]):
    wb.check(f"7.4{letter}", answer)

**Ex 7.5 — Le rayon de l'hyper-orange : r(d) = √d − 1**

In [ ]:
# Ex 7.5: the values you found on paper
answer_7_5a = ...  # a) r(2), 3 decimals
answer_7_5b = ...  # b) r(3), 3 decimals
answer_7_5c = ...  # c) a whole number
answer_7_5d = ...  # d) the smallest dimension
answer_7_5e = ...  # e) the smallest dimension
answer_7_5f = ...  # f) r(50), 3 decimals

for letter, answer in zip("abcdef", [answer_7_5a, answer_7_5b, answer_7_5c, answer_7_5d, answer_7_5e, answer_7_5f]):
    wb.check(f"7.5{letter}", answer)

**Ex 7.6 — Boule dans un cube : rapport des volumes par récurrence**

In [ ]:
# Ex 7.6: the values you found on paper
answer_7_6a = ...  # a) V4, 3 decimals
answer_7_6b = ...  # b) V5, 3 decimals
answer_7_6c = ...  # c) q4, 3 decimals
answer_7_6d = ...  # d) q10 in percent, 3 decimals
answer_7_6e = ...  # e) a dimension
answer_7_6f = ...  # f) 3 decimals

for letter, answer in zip("abcdef", [answer_7_6a, answer_7_6b, answer_7_6c, answer_7_6d, answer_7_6e, answer_7_6f]):
    wb.check(f"7.6{letter}", answer)

## Partie A · Frontières, distances, centroïde le plus proche, carte de probabilité, un-contre-tous

Fiche §7.2 à §7.5 (jusqu'à « Combien de clusters ? »). Tu entraînes un premier classifieur sur des œufs imaginaires, tu prévois où k-means coupera deux lunes, tu écris tes premières fonctions de `mylearn.cluster` (les distances, puis le classifieur du centroïde le plus proche), tu dessines une carte de probabilité avec deux seuils, et tu prévois ce que l'un-contre-tous fait de quatre classes. Dans tous les exercices de code, donne les valeurs calculées **sans les arrondir** : la vérification arrondit elle-même (sauf quand l'énoncé fixe un nombre de décimales). La cellule ci-dessous charge les outils de tout le notebook : exécute-la d'abord.

In [ ]:
# Tools for the notebook exercises (parts A to D)
import copy
import itertools
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
from scipy.spatial.distance import pdist
from threadpoolctl import threadpool_limits
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import silhouette_score as sklearn_silhouette_score
from sklearn.neighbors import NearestCentroid as SklearnNearestCentroid


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def fr(x, digits=3):
    """A number written the French way, with a decimal comma (for the messages in French)."""
    return f"{x:.{digits}f}".replace(".", ",")


def returned(ex_id, name, value):
    """False, with a ❌ message, when the function `name` returned None (a forgotten return); True otherwise."""
    if value is None:
        print(f"❌ Ex {ex_id} : {name} renvoie None : as-tu oublié le return ?")
        return False
    return True


def fitted(model, X, y=None):
    """The model after model.fit(X) or model.fit(X, y): no chained call, so that a fit that forgets
    `return self` (the tests report it) does not stop the check cells."""
    if y is None:
        model.fit(X)
    else:
        model.fit(X, y)
    return model


def run_mylearn_tests(module, keyword, impl="learner"):
    """Run the tests of mylearn.<module> ("cluster" or "multiclass") selected by `keyword` (on YOUR code by default)."""
    test_file = f"tests/test_ch07_{module}.py"
    command = [sys.executable, "-m", "pytest", test_file, "-k", keyword, "-q", "-p", "no:cacheprovider",
               "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    failed.sort(key=lambda line: "returns_self" not in line and "copies" not in line)   # the clearest tests first
    if any("'NoneType' object has no attribute" in line for line in failed):
        print("💡 un attribut est lu sur None : une méthode (souvent fit) oublie-t-elle son return (return self) ?")
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix(f"FAILED {test_file}::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


def show_clusters(ax, X, labels, title="", centres=None):
    """Scatter the 2-D points X, one colour per label (-1, the noise of DBSCAN and HDBSCAN, in grey)."""
    labels = np.asarray(labels)
    colours = wb.plot.class_colors(max(len(set(labels.tolist()) - {-1}), 1))
    for i, label in enumerate(sorted(set(labels.tolist()) - {-1})):
        ax.scatter(*X[labels == label].T, s=12, color=colours[i % len(colours)], label=f"cluster {label}")
    if (labels == -1).any():
        ax.scatter(*X[labels == -1].T, s=12, color="lightgray", label="noise (-1)")
    if centres is not None:
        ax.scatter(*np.asarray(centres).T, marker="X", s=200, color="black", label="centres")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="datalim")

### Ex 7.11 — Des œufs en 2D : données, régions et frontière de décision 📦 ★ ⏱️ 15 min
**Objectif :** entraîner un premier classifieur avec scikit-learn, dessiner ses régions de décision et mesurer sa généralisation.  
**Prérequis :** fiche §7.2, §7.2.1 et l'encadré 🧮 sur le centroïde le plus proche (§7.5) · ch. 1 (entraînement et test) · **Fil rouge :** synthétique (œufs) · **Parcours :** R, C

`make_eggs(n, seed)` (fourni) fabrique des œufs imaginaires, décrits par deux features : le poids (en g) et la longueur (en mm). Le label vaut 1 si l'œuf est fécondé, 0 sinon. Les nombres sont inventés : le livre ne donne aucune donnée, c'est son expérience de pensée du §7.2.1, en vrai.

a) `n_fertile_11` : combien des 200 œufs d'entraînement (`X_11`, `y_11`) sont fécondés ?
b) entraîne `model_11 = SklearnNearestCentroid()`, le classifieur du centroïde le plus proche de scikit-learn (`sklearn.neighbors.NearestCentroid`), sur ces œufs (`fit`) ; `train_acc_11` : son accuracy sur ces mêmes œufs (`model_11.score`) ;
c) `new_acc_11` : son accuracy sur 400 œufs nouveaux (`X_new_11`, `y_new_11`) ;
d) `pred_11` : ses prédictions pour les trois œufs de `EGGS_11`, sous forme de liste (`.tolist()`).

La dernière cellule dessine les deux régions de décision et les trois œufs de d).

Dans tes notes : quelle forme a la frontière ? Où sont les œufs mal classés ? Les accuracies de b) et c) diffèrent : laquelle t'intéresse vraiment, et pourquoi ?

In [ ]:
def make_eggs(n, seed):
    """n imaginary eggs: weight (g), length (mm) and label (1 = fertilised, 0 = not), two Gaussian clouds."""
    rng = np.random.default_rng(seed)
    y = (rng.random(n) < 0.5).astype(int)
    mean = np.where(y[:, None] == 1, [62.0, 56.0], [57.0, 58.5])     # fertilised eggs: heavier and shorter
    X = mean + rng.normal(0.0, [2.2, 1.2], (n, 2))
    return X, y


X_11, y_11 = make_eggs(200, seed=723)               # the eggs of the farm, candled by an expert (the labels)
X_new_11, y_new_11 = make_eggs(400, seed=1723)      # new eggs, never seen during training
EGGS_11 = np.array([[55.5, 59.5], [64.0, 55.0], [58.0, 59.0]])   # three eggs to classify in d)
print("first eggs (weight, length):", X_11[:3].round(1).tolist(), "· labels:", y_11[:3].tolist())

In [ ]:
n_fertile_11 = ...   # a) an integer
model_11 = ...       # b) SklearnNearestCentroid(), fitted on X_11, y_11
train_acc_11 = ...   # b) its accuracy on X_11, y_11
new_acc_11 = ...     # c) its accuracy on X_new_11, y_new_11
pred_11 = ...        # d) the predictions for EGGS_11, as a list

In [ ]:
wb.check("7.11a", n_fertile_11)
wb.check("7.11b", train_acc_11)
wb.check("7.11c", new_acc_11)
wb.check("7.11d", pred_11)

**Les régions de décision** : exécute la cellule une fois `model_11` entraîné.

In [ ]:
if filled(model_11):
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    wb.plot.plot_decision_boundary(model_11, X_11, y_11, ax=ax, class_names=["not fertilised (0)", "fertilised (1)"])
    ax.scatter(EGGS_11[:, 0], EGGS_11[:, 1], marker="*", s=300, color="black", zorder=5, label="the eggs of d)")
    ax.set(xlabel="weight (g)", ylabel="length (mm)", title="Nearest centroid: the two decision regions")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)   # outside: its star is not an egg
    plt.show()
else:
    print("⏳ Ex 7.11 : entraîne d'abord model_11 (question b), puis relance cette cellule.")

### Ex 7.12 — k-means sur deux lunes : où tombera la coupure ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir la forme des clusters de k-means sur deux groupes emboîtés, avant de lire la section qui l'explique.  
**Prérequis :** fiche §7.5 jusqu'à « Combien de clusters ? » (pas encore « Quand k-means échoue ») · Ex 7.11 · **Fil rouge :** synthétique (lunes) · **Parcours :** R, C

Fais cet exercice **avant** de lire la section « Quand k-means échoue » de la fiche. Les 300 points de `X_12` forment deux lunes entrelacées (`wb.synth.make_moons`). On lance k-means avec $k = 2$ (`SklearnKMeans`, scikit-learn), qui ne voit que les points, pas leur lune.

Prédis, dans la cellule ci-dessous, **avant** d'exécuter l'expérience :
a) `shape_12` : la frontière entre les deux clusters sera `"A"` une courbe qui suit les lunes, `"B"` une droite, ou `"C"` un cercle ;
b) `agree_12` : la part des points rangés dans le même cluster que les autres points de leur lune (au mieux des deux numérotations des clusters) sera proche de `0.5`, `0.75` ou `1.0`.

Puis exécute l'expérience : elle colore les clusters, marque les deux centres et affiche la part de b).

Dans tes notes : compare avec tes prédictions. Avec l'algorithme de Lloyd (fiche, encadré 🧮), explique la forme de la frontière : que fait l'étape d'affectation de chaque point du plan ? Lis ensuite la section « Quand k-means échoue ».

In [ ]:
X_12, moon_12 = wb.synth.make_moons(n=300, noise=0.06, seed=712)   # moon_12: the moon each point comes from
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.scatter(X_12[:, 0], X_12[:, 1], s=12, color="gray")
ax.set(title="300 points, shown without their moon", aspect="equal")
plt.show()

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
shape_12 = ...   # a) "A" (a curve that follows the moons), "B" (a straight line) or "C" (a circle)
agree_12 = ...   # b) 0.5, 0.75 or 1.0

In [ ]:
if any(answer is ... or answer is None for answer in [shape_12, agree_12]):
    print("⏳ Ex 7.12 : écris d'abord tes deux prédictions, puis relance cette cellule.")
else:
    km_12 = SklearnKMeans(n_clusters=2, n_init=10, random_state=0).fit(X_12)
    labels_12 = km_12.labels_
    agreement_12 = max(np.mean(labels_12 == moon_12), np.mean(labels_12 != moon_12))   # the better of the two numberings
    fig, ax = plt.subplots(figsize=(6, 3.8))
    wb.plot.plot_decision_boundary(km_12, X_12, labels_12, ax=ax, class_names=["cluster 0", "cluster 1"])
    ax.scatter(*km_12.cluster_centers_.T, marker="X", s=220, color="black", zorder=5, label="centres")
    ax.set(title="k-means with k = 2 on the two moons", aspect="equal")
    ax.legend(fontsize=8)
    plt.show()
    print(f"share of the points clustered with their own moon: {agreement_12:.3f}")
    print(f"your predictions: a) {shape_12}   b) {agree_12}")

### Ex 7.13 — Distances au carré vectorisées : pairwise_sq_distances 🔨 ★★ ⏱️ 20 min
**Objectif :** calculer toutes les distances au carré entre deux nuages de points sans boucle Python.  
**Prérequis :** rappel 7.R3 · 0A (broadcasting, réductions par axe) · fiche §7.5 (encadré 🧮 « toutes les distances d'un coup ») · **Fil rouge :** synthétique · **mylearn :** `cluster.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/cluster.py` (créé par `python tools/start_chapter.py 7`), lis la docstring de chaque fonction, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy est permis ; SciPy et scikit-learn non : `scipy.spatial.distance.cdist` et les classes de scikit-learn sont les **oracles** des tests. La cellule de vérification recharge ta librairie, affiche quelques valeurs, puis lance les tests de tes fonctions ; `python -m pytest tests/test_ch07_cluster.py -q` les lance tous dans un terminal.

Écris `pairwise_sq_distances(A, B)` (lis sa docstring) :
- convertis `A` et `B` en tableaux de flottants (`np.asarray(..., dtype=float)`), et lève une `ValueError` si l'un d'eux n'est pas à deux dimensions, ou s'ils n'ont pas le même nombre de colonnes ;
- calcule toutes les distances au carré **sans boucle Python**, d'une de ces deux façons : par broadcasting, `A[:, None, :] - B[None, :, :]` donne toutes les différences, un tableau de forme $(n_a, n_b, d)$ dont tu sommes les carrés sur le dernier axe ; ou par l'identité du rappel 7.R3, avec les normes au carré des lignes (`(A ** 2).sum(axis=1)`), **un** produit matriciel `A @ B.T` et le broadcasting ;
- remplace les minuscules valeurs négatives dues aux arrondis par 0 (`np.maximum(D, 0.0)`).

Les deux façons sont justes, mais garde de préférence l'identité : tu réutiliseras ta fonction sur des images de 784 pixels (7.20), où le tableau intermédiaire du broadcasting pèserait plus d'un gigaoctet.

La cellule de vérification affiche l'exemple de la docstring, compare ta fonction à une double boucle Python et mesure leurs vitesses, puis lance les tests.

Dans tes notes : pour 10 000 points contre 10 000 points en dimension 784, combien de nombres chaque méthode stocke-t-elle au plus fort du calcul ? Pourquoi l'identité peut-elle donner des valeurs négatives, et pourquoi faut-il s'en soucier avant de prendre une racine carrée ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.13"):
    A_13 = np.array([[0.0, 0.0], [1.0, 1.0]])
    B_13 = np.array([[1.0, 0.0], [0.0, 2.0], [3.0, 4.0]])
    print("the example of the docstring:\n", mylearn.cluster.pairwise_sq_distances(A_13, B_13))
    rng_13 = np.random.default_rng(713)
    P_13, Q_13 = rng_13.normal(size=(300, 5)), rng_13.normal(size=(200, 5))
    start_13 = time.perf_counter()
    loop_13 = np.array([[((p - q) ** 2).sum() for q in Q_13] for p in P_13])     # 60 000 Python iterations
    loop_time_13 = time.perf_counter() - start_13
    start_13 = time.perf_counter()
    fast_13 = mylearn.cluster.pairwise_sq_distances(P_13, Q_13)
    fast_time_13 = time.perf_counter() - start_13
    if returned("7.13", "pairwise_sq_distances", fast_13):
        verdict("7.13", np.shape(fast_13) == (300, 200) and np.allclose(fast_13, loop_13),
                "les mêmes distances que la double boucle.",
                "tes distances ne sont pas celles de la double boucle (forme (300, 200) attendue) : relis la docstring.")
        print(f"double loop: {loop_time_13 * 1000:.0f} ms · yours: {fast_time_13 * 1000:.2f} ms, "
              f"about {loop_time_13 / max(fast_time_13, 1e-9):.0f} times faster")
    run_mylearn_tests("cluster", "test_pairwise_sq_distances_")

### Ex 7.14 — Le classifieur du centroïde le plus proche 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire un premier classifieur « à la scikit-learn » : fit, decision_function, predict et score.  
**Prérequis :** Ex 7.13 · ∂ 7.7 · fiche §7.3 et §7.5 (encadré 🧮 sur le centroïde le plus proche) · **Fil rouge :** synthétique (œufs) · **mylearn :** `cluster.py` · **Parcours :** R, C

> **mylearn** : dans `mon_travail/mylearn/cluster.py`, mêmes règles qu'en 7.13 (NumPy permis, SciPy et scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris la classe `NearestCentroid` (lis ses docstrings). C'est ta première classe « à la scikit-learn » : `__init__` ne fait rien (pas d'hyperparamètre), `fit` apprend et renvoie `self`, et les attributs appris finissent par `_`.
- `fit(X, y)` : `classes_` est le tableau **trié** des labels distincts (`np.unique`) ; `centroids_[k]` est la moyenne des lignes de `X` de la classe `classes_[k]`, dans le même ordre. Lève une `ValueError` s'il y a moins de deux classes, ou si `X` et `y` n'ont pas le même nombre de lignes ;
- `decision_function(X)` : réutilise ta `pairwise_sq_distances` (7.13). Avec deux classes, un score par ligne, $d^2(\mathbf{x}, \boldsymbol{\mu}_0) - d^2(\mathbf{x}, \boldsymbol{\mu}_1)$, positif quand la classe `classes_[1]` est la plus proche ; avec $K > 2$ classes, une colonne par classe, $-d^2(\mathbf{x}, \boldsymbol{\mu}_k)$ ;
- `predict(X)` : le label du centroïde le plus proche. En cas d'égalité de distances, **la première classe** de `classes_` gagne : c'est ce que fait `np.argmin`, qui garde le premier minimum ;
- `score(X, y)` : l'accuracy de `predict(X)`, un `float` Python.

Puis la vérification entraîne ta classe sur les trois classes d'œufs du livre (§7.3 : `"viable"`, `"clair"` pour le *yolker*, `"mort"` pour le *quitter*) et vérifie :
a) son accuracy sur 600 œufs nouveaux ;
b) le centroïde de la classe `"mort"` ;
enfin, elle compare tes prédictions à celles de scikit-learn, dessine les trois régions et lance les tests.

Dans tes notes : pourquoi les frontières sont-elles des morceaux de droites (∂ 7.7) ? Quelles classes se confondent le plus, et pourquoi ?

In [ ]:
def make_egg_classes(n, seed):
    """n imaginary eggs of the three classes of the book (§7.3): weight (g), length (mm) and class name."""
    rng = np.random.default_rng(seed)
    names = np.array(["viable", "clair", "mort"])      # fertilised and alive / never fertilised / dead embryo
    means = np.array([[62.0, 56.0], [56.5, 58.5], [60.0, 59.5]])
    which = rng.integers(0, 3, n)
    X = means[which] + rng.normal(0.0, [1.8, 1.0], (n, 2))
    return X, names[which]


X_14, y_14 = make_egg_classes(300, seed=714)
X_new_14, y_new_14 = make_egg_classes(600, seed=1714)
print({str(name): int(count) for name, count in zip(*np.unique(y_14, return_counts=True))})

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.14"):
    X_doc_14 = np.array([[0.0, 0.0], [0.0, 2.0], [4.0, 0.0], [4.0, 2.0]])
    doc_14 = fitted(mylearn.cluster.NearestCentroid(), X_doc_14, np.array([0, 0, 1, 1]))
    print("docstring example: centroids", np.asarray(doc_14.centroids_).tolist(), "· decision_function",
          np.asarray(doc_14.decision_function(np.array([[1.0, 1.0], [3.5, 0.0]]))).tolist())
    model_14 = fitted(mylearn.cluster.NearestCentroid(), X_14, y_14)
    classes_14 = np.asarray(model_14.classes_).tolist()
    print("classes_:", classes_14)
    score_14 = model_14.score(X_new_14, y_new_14)
    if returned("7.14", "score", score_14):
        wb.check("7.14a", score_14, computed=True)
    if "mort" in classes_14:
        wb.check("7.14b", np.asarray(model_14.centroids_)[classes_14.index("mort")], computed=True)
    else:
        print("❌ Ex 7.14 : classes_ doit contenir les labels eux-mêmes, triés : ['clair', 'mort', 'viable'].")
    same_14 = np.array_equal(np.asarray(model_14.predict(X_new_14)),
                             SklearnNearestCentroid().fit(X_14, y_14).predict(X_new_14))
    verdict("7.14", same_14, "les mêmes prédictions que NearestCentroid de scikit-learn.",
            "tes prédictions diffèrent de celles de scikit-learn sur les œufs nouveaux.")
    if same_14:
        fig, ax = plt.subplots(figsize=(6.5, 4.6))
        wb.plot.plot_decision_boundary(model_14, X_new_14, y_new_14, ax=ax, class_names=classes_14)
        ax.scatter(*np.asarray(model_14.centroids_).T, marker="X", s=220, color="black", zorder=5, label="centroids")
        ax.set(xlabel="weight (g)", ylabel="length (mm)", title="Three classes of eggs: the regions of the nearest centroid")
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
        plt.show()
    run_mylearn_tests("cluster", "test_nearest_centroid_")

### Ex 7.15 — Carte de probabilité et politique de seuil pour les œufs 📈 ★★ ⏱️ 25 min
**Objectif :** calculer une carte de probabilité par la règle de Bayes, y lire l'effet d'un seuil, et chiffrer le coût de deux politiques.  
**Prérequis :** rappel 7.R2 · ch. 2 (loi normale) · ch. 3 (seuil, faux positifs et faux négatifs) · fiche §7.2.1 (encadré 🧮 sur le seuil de coût minimal) · **Fil rouge :** synthétique (œufs) · **Parcours :** M, C

Dans une autre ferme, 35 % des œufs sont fécondés (`PRIOR_15`). Les mesures (poids, longueur) suivent une loi normale à deux dimensions dans chaque classe, de moyennes et de matrices de covariance données (`MEAN_1_15`, `COV_1_15` pour les œufs fécondés, `MEAN_0_15`, `COV_0_15` pour les autres). `scipy.stats.multivariate_normal(mean, cov).pdf(X)` donne la densité de cette loi en chaque ligne de `X` (ch. 2 : c'est la courbe en cloche, en deux dimensions). Garder un œuf non fécondé dans l'incubateur coûte 1 (un faux positif), vendre un œuf fécondé coûte 6 (un faux négatif).

a) écris `posterior_15(X)`, qui renvoie $P(\text{fécondé} \mid \mathbf{x})$ pour chaque ligne de `X` par la règle de Bayes (fiche §7.2.1, rappel 7.R2) ; la vérification l'appelle sur les quatre œufs A, B, C, D de `POINTS_15` ;
b) `t_star_15` : le seuil qui minimise le coût moyen (fiche, encadré 🧮), sous forme de calcul ou avec 3 décimales.

Une cellule plus bas, après la vérification, dessine la carte de $P(\text{fécondé} \mid \mathbf{x})$, ses deux lignes de niveau (au seuil 0,5 et au seuil $t^*$) et les quatre œufs. **Lis la carte**, puis réponds :
c) `changes_15` : l'ensemble des lettres des œufs dont la décision change quand on passe du seuil 0,5 au seuil $t^*$ (par exemple `{"X", "Y"}`) ;
d) `counts_15` : sur les 2 000 œufs simulés `eggs_15` (dont la vérité est `truth_15`), déclare « fécondé » quand $P \ge t$ ; donne `[[FN, FP] au seuil 0,5, [FN, FP] au seuil t*]` ;
e) `costs_15` : le coût moyen par œuf à chacun des deux seuils, `[coût à 0,5, coût à t*]`, avec les coûts de l'énoncé.

Dans tes notes : pourquoi la frontière est-elle courbe, alors que celle du centroïde le plus proche est une droite ? Qui devrait choisir les deux coûts (⚖️ 7.10) ?

In [ ]:
PRIOR_15 = 0.35                                    # share of fertilised eggs in this farm
MEAN_1_15, COV_1_15 = np.array([61.0, 56.0]), np.array([[4.0, 1.2], [1.2, 1.5]])     # fertilised eggs
MEAN_0_15, COV_0_15 = np.array([57.0, 58.0]), np.array([[5.0, -1.0], [-1.0, 2.0]])   # non-fertilised eggs
COST_FP_15, COST_FN_15 = 1.0, 6.0                  # keep a non-fertilised egg (FP) / sell a fertilised one (FN)
POINTS_15 = {"A": (62.0, 55.5), "B": (59.5, 57.0), "C": (58.5, 56.5), "D": (59.2, 58.0)}   # four eggs to read

rng_15 = np.random.default_rng(715)                # 2000 simulated eggs of this farm, with their truth
truth_15 = (rng_15.random(2000) < PRIOR_15).astype(int)
eggs_15 = np.where(truth_15[:, None] == 1, rng_15.multivariate_normal(MEAN_1_15, COV_1_15, 2000),
                   rng_15.multivariate_normal(MEAN_0_15, COV_0_15, 2000))
print(f"{truth_15.sum()} fertilised eggs among {len(truth_15)}")

In [ ]:
def posterior_15(X):
    """P(fertilised | x) for every row x of X, by Bayes' rule: an array of shape (n,)."""
    raise NotImplementedError("posterior_15() is not written yet")


t_star_15 = ...   # b) the threshold that minimises the mean cost

In [ ]:
with wb.attempt("7.15"):
    p_points_15 = posterior_15(np.array(list(POINTS_15.values())))
    if returned("7.15", "posterior_15", p_points_15):
        print({name: round(float(p), 3) for name, p in zip(POINTS_15, np.ravel(p_points_15))})
        wb.check("7.15a", p_points_15, computed=True)
wb.check("7.15b", t_star_15)

**La carte** : exécute la cellule une fois `posterior_15` et `t_star_15` écrits.

In [ ]:
with wb.attempt("7.15"):
    if not filled(t_star_15):
        raise NotImplementedError("write t_star_15 first (question b)")
    weights_15, lengths_15 = np.meshgrid(np.linspace(51, 68, 240), np.linspace(52.5, 61.5, 200))
    grid_15 = np.column_stack([weights_15.ravel(), lengths_15.ravel()])
    p_grid_15 = posterior_15(grid_15)
    if returned("7.15", "posterior_15", p_grid_15):
        p_grid_15 = np.asarray(p_grid_15, dtype=float).reshape(weights_15.shape)
        fig, ax = plt.subplots(figsize=(7.5, 4.8))
        shade_15 = ax.contourf(weights_15, lengths_15, p_grid_15, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.75)
        plt.colorbar(shade_15, ax=ax, label="P(fertilised | x)")
        lines_15 = ax.contour(weights_15, lengths_15, p_grid_15, levels=sorted([t_star_15, 0.5]), colors="black",
                              linestyles=["--", "-"] if t_star_15 < 0.5 else ["-", "--"], linewidths=1.6)
        ax.clabel(lines_15, fmt=lambda level: f"P = {level:.3f}".rstrip("0"), fontsize=8)
        for name, (w, l) in POINTS_15.items():
            ax.scatter(w, l, marker="o", s=60, color="black", zorder=5)
            ax.annotate(name, (w, l), xytext=(5, 5), textcoords="offset points", fontsize=12, weight="bold")
        ax.set(xlabel="weight (g)", ylabel="length (mm)", title="Probability of being fertilised, and two thresholds")
        plt.show()

Lis la carte, puis réponds aux questions c) à e).

In [ ]:
changes_15 = ...   # c) a set of letters
counts_15 = ...    # d) [[FN, FP] at the threshold 0.5, [FN, FP] at t_star_15]
costs_15 = ...     # e) [mean cost per egg at 0.5, mean cost per egg at t_star_15]

In [ ]:
wb.check("7.15c", changes_15)
wb.check("7.15d", counts_15)
wb.check("7.15e", costs_15)

### Ex 7.16 — Un-contre-tous avec des centroïdes : quelle classe sera sacrifiée ? 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir quelle classe l'un-contre-tous traite le plus mal, et pourquoi.  
**Prérequis :** Ex 7.14 · fiche §7.4.1 · **Fil rouge :** synthétique · **Parcours :** C

Quatre classes A, B, C et D (figure ci-dessous). On les classe de deux façons, toutes deux avec **ton** `NearestCentroid` (7.14) :
- le **centroïde le plus proche multi-classe** : un seul modèle, quatre centroïdes ;
- l'**un-contre-tous** (fiche §7.4.1) : pour chaque classe $k$, un `NearestCentroid` binaire apprend « $k$ » (label 1) contre « tout le reste » (label 0). Il a donc deux centroïdes : celui de la classe $k$, et celui de **toutes les autres classes réunies**. Son score, `decision_function`, est positif quand le centroïde de $k$ est le plus proche ; on prédit la classe du plus grand des quatre scores.

On compare le **recall** de chaque classe (la part de ses points de test bien classés) dans les deux cas.

Prédis, **avant** d'exécuter l'expérience :
a) `sacrificed_16` : la lettre de la classe dont le recall baissera le plus avec l'un-contre-tous ;
b) `recall_16` : son recall avec l'un-contre-tous sera proche de `0.9`, `0.6` ou `0.3` ?

Indice pour raisonner : pour chaque classe, où tombe le centroïde de « tout le reste » ?

Dans tes notes : compare avec tes prédictions, et explique ce qui arrive au classifieur binaire de la classe sacrifiée. Tu écriras en 7.22 un un-contre-tous générique, qui refera ce calcul pour n'importe quel classifieur binaire.

In [ ]:
LAYOUT_16 = {"A": (-4.0, -2.0), "B": (4.0, -2.0), "C": (0.0, 5.0), "D": (0.0, 0.33)}   # the centre of each class


def make_four_16(n, seed):
    """n points of each of the four classes of LAYOUT_16 (standard deviation 1.4)."""
    rng = np.random.default_rng(seed)
    X = np.vstack([rng.normal(centre, 1.4, (n, 2)) for centre in LAYOUT_16.values()])
    return X, np.repeat(np.array(list(LAYOUT_16)), n)


X_16, y_16 = make_four_16(100, seed=716)            # training points
X_test_16, y_test_16 = make_four_16(300, seed=1716)  # test points
fig, ax = plt.subplots(figsize=(5, 4.2))
for name in LAYOUT_16:
    ax.scatter(*X_16[y_16 == name].T, s=12, label=name)
ax.set(title="Four classes (training points)", aspect="equal")
ax.legend()
plt.show()

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
sacrificed_16 = ...   # a) "A", "B", "C" or "D"
recall_16 = ...       # b) 0.9, 0.6 or 0.3

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
if any(answer is ... or answer is None for answer in [sacrificed_16, recall_16]):
    print("⏳ Ex 7.16 : écris d'abord tes deux prédictions, puis relance cette cellule.")
else:
    with wb.attempt("7.16"):
        classes_16 = np.unique(y_16)
        native_16 = fitted(mylearn.cluster.NearestCentroid(), X_16, y_16)          # one multi-class model
        binary_16 = [fitted(mylearn.cluster.NearestCentroid(), X_16, (y_16 == c).astype(int))
                     for c in classes_16]                                           # "c against the rest", for each c


        def by_hand_16(points):
            """One-versus-rest by hand: the class of the highest of the four binary scores."""
            return classes_16[np.column_stack([model.decision_function(points) for model in binary_16]).argmax(axis=1)]


        ovr_16 = by_hand_16(X_test_16)
        pred_native_16 = np.asarray(native_16.predict(X_test_16))
        print("recall    " + "  ".join(f"{c:>5}" for c in classes_16))
        print("native NC " + "  ".join(f"{np.mean(pred_native_16[y_test_16 == c] == c):5.3f}" for c in classes_16))
        print("OvR       " + "  ".join(f"{np.mean(ovr_16[y_test_16 == c] == c):5.3f}" for c in classes_16))
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
        wb.plot.plot_decision_boundary(native_16, X_test_16, y_test_16, ax=axes[0], class_names=list(classes_16))
        axes[0].set_title("multi-class nearest centroid")
        wb.plot.plot_decision_boundary(by_hand_16, X_test_16, y_test_16, ax=axes[1], class_names=list(classes_16))
        axes[1].set_title("one-versus-rest with nearest centroids")
        plt.show()
        print(f"your predictions: a) {sacrificed_16}   b) {recall_16}")

## Partie B · k-means, DBSCAN et HDBSCAN avec scikit-learn, puis la grande dimension

Fiche §7.5 (« Quand k-means échoue ») et §7.6, puis §7.6.1 **après** le 🔮 7.19. Tu cherches les espèces de manchots avec k-means, tu compares k-means au clustering par densité sur des formes quelconques, puis tu mesures ce que la dimension fait aux densités et aux distances. La cellule ci-dessous charge les manchots du ch. 1.

In [ ]:
penguins = wb.datasets.load_penguins(dropna=True)          # ch. 1: the 333 complete penguins
FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X_peng = penguins[FEATURES].to_numpy(dtype=float)
species = penguins["species"].to_numpy()                   # never shown to k-means: only used to judge it
order_peng = np.random.default_rng(42).permutation(len(penguins))   # the split of ch. 1
TEST_PENG, TRAIN_PENG = order_peng[:100], order_peng[100:]
print(X_peng.shape, {str(name): int(count) for name, count in zip(*np.unique(species, return_counts=True))})

### Ex 7.17 — Manchots sans labels : k-means face aux espèces 📦 ★★ ⏱️ 25 min
**Objectif :** regrouper des données réelles avec k-means, mesurer l'effet de la standardisation et juger les clusters avec des labels gardés de côté.  
**Prérequis :** ch. 2 (z-scores) · ch. 3 (`pd.crosstab`) · fiche §7.5 (pureté, « Quand k-means échoue ») · **Fil rouge :** Penguins · **Parcours :** R, C

Les 333 manchots complets du ch. 1 sont décrits par quatre mesures (`X_peng` : longueur et épaisseur du bec en mm, longueur de la nageoire en mm, masse en g). Leur espèce (`species`) n'est **jamais** montrée à k-means : elle sert seulement à juger les clusters trouvés.

a) écris `purity_17(labels, truth)`, la **pureté** d'un clustering (fiche §7.5) : chaque cluster prend sa classe la plus fréquente, et la pureté est la part des points qui portent cette classe. `pd.crosstab(labels, truth)` (ch. 3) donne le tableau des effectifs, une ligne par cluster. La vérification l'essaie sur un petit découpage (`TOY_LABELS_17`, `TOY_TRUTH_17`) ;
b) `raw_labels_17` : les clusters de `SklearnKMeans(n_clusters=3, n_init=10, random_state=0)` sur les mesures **brutes** `X_peng` (`.fit(...).labels_`) ; `raw_purity_17` : leur pureté ;
c) `Z_17` : les mesures **standardisées**, colonne par colonne (z-scores du ch. 2, avec la moyenne et l'écart-type de NumPy, `ddof=0`) ; `std_labels_17` et `std_purity_17` : les mêmes questions sur `Z_17` ;
d) `ari_17` : l'**indice de Rand ajusté** de c), `adjusted_rand_score(species, std_labels_17)`. Il vaut 1 pour deux découpages identiques et environ 0 pour un découpage au hasard, et il ne dépend pas de la numérotation des clusters ;
e) `split_17` : dans le découpage de c), le nom de l'espèce **la plus partagée** entre plusieurs clusters, celle dont la plus petite part des manchots se trouve dans son cluster principal (lis le tableau croisé affiché).

Dans tes notes : pourquoi les mesures brutes donnent-elles de si mauvais clusters ? Quelle mesure domine les distances, et pourquoi ? Un cluster est-il une espèce ?

In [ ]:
TOY_LABELS_17 = np.array([0, 0, 0, 1, 1, 2, 2, 2])                      # a small clustering, to test purity_17
TOY_TRUTH_17 = np.array(["x", "x", "y", "x", "x", "y", "y", "z"])

In [ ]:
def purity_17(labels, truth):
    """Purity of a clustering: each cluster takes its most frequent true class; share of the points that carry it."""
    raise NotImplementedError("purity_17() is not written yet")


raw_labels_17 = ...    # b) the clusters of SklearnKMeans(n_clusters=3, n_init=10, random_state=0) on X_peng
raw_purity_17 = ...    # b) their purity
Z_17 = ...             # c) X_peng standardised, column by column
std_labels_17 = ...    # c) the clusters of the same KMeans on Z_17
std_purity_17 = ...    # c) their purity
ari_17 = ...           # d) adjusted_rand_score(species, std_labels_17)
split_17 = ...         # e) the name of the species shared the most between clusters, in c)

In [ ]:
with wb.attempt("7.17"):
    toy_17 = purity_17(TOY_LABELS_17, TOY_TRUTH_17)
    if returned("7.17", "purity_17", toy_17):
        wb.check("7.17a", toy_17, computed=True)
wb.check("7.17b", raw_purity_17)
wb.check("7.17c", std_purity_17)
wb.check("7.17d", ari_17)
wb.check("7.17e", split_17)
if filled(std_labels_17):
    print(pd.crosstab(std_labels_17, species, rownames=["cluster"], colnames=["species"]))

### Ex 7.18 — Formes arbitraires et bruit : DBSCAN et HDBSCAN 📦 ★★ ⏱️ 25 min
**Objectif :** comparer k-means et le clustering par densité sur des groupes de formes et de densités différentes, avec du bruit.  
**Prérequis :** Ex 7.12 · Ex 7.17 (indice de Rand ajusté) · fiche §7.5 (« Quand k-means échoue », 🕰️ sur DBSCAN et HDBSCAN) · **Fil rouge :** synthétique · **Parcours :** R, C

`X_18` réunit quatre groupes : deux lunes (300 points), un grand groupe étalé (100 points), un petit groupe très serré (60 points), et 30 points dispersés, du bruit. `group_18` donne la vérité (−1 pour le bruit) : les algorithmes ne la voient pas.

a) écris `summary_18(labels)`, qui renvoie le triplet `(nombre de clusters, nombre de points de bruit, ARI)` : on ne compte pas −1 comme un cluster, et l'ARI (`adjusted_rand_score`, 7.17) compare `labels` à `group_18` sur les seuls points qui ne sont **pas** du bruit dans `group_18` ;
b) `km_labels_18` : les clusters de `SklearnKMeans(n_clusters=4, n_init=10, random_state=0)` ;
c) `db_labels_18` : ceux de `DBSCAN(eps=0.2, min_samples=5)` (`.fit_predict(X_18)` ; −1 marque le bruit) ;
d) `scan_18` : un dictionnaire `{eps: summary_18(labels)}` pour chaque `eps` de `EPS_18`, avec `min_samples=5` ;
e) `hdb_labels_18` : ceux de `HDBSCAN(min_cluster_size=10)`, qui n'a pas d'`eps`.

La vérification affiche les tableaux et les trois découpages.

Dans tes notes : pourquoi k-means échoue-t-il, même avec le bon nombre de groupes ? Pourquoi aucun `eps` de DBSCAN ne convient-il à la fois aux deux lunes, denses et proches l'une de l'autre, et au grand groupe étalé ? Que fait HDBSCAN de différent (fiche, 🕰️) ? Lesquels des trois algorithmes ont une méthode `predict` pour un point nouveau ?

In [ ]:
moons_18, moon_label_18 = wb.synth.make_moons(n=300, noise=0.05, seed=718)
rng_18 = np.random.default_rng(718)
X_18 = np.vstack([moons_18,
                  rng_18.normal((4.0, 0.8), 0.45, (100, 2)),          # a wide, sparse blob
                  rng_18.normal((4.0, -1.2), 0.08, (60, 2)),          # a small, very dense blob
                  rng_18.uniform([-1.5, -2.0], [5.5, 2.0], (30, 2))])  # 30 scattered points: noise
group_18 = np.concatenate([moon_label_18, np.full(100, 2), np.full(60, 3), np.full(30, -1)])   # -1: noise
EPS_18 = [0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
fig, ax = plt.subplots(figsize=(7, 3.8))
show_clusters(ax, X_18, group_18, "The four groups and the noise (the truth, unknown to the algorithms)")
ax.legend(fontsize=7, loc="lower left")
plt.show()

In [ ]:
def summary_18(labels):
    """(number of clusters, number of noise points, ARI with group_18 on its non-noise points)."""
    raise NotImplementedError("summary_18() is not written yet")


km_labels_18 = ...    # b) SklearnKMeans(n_clusters=4, n_init=10, random_state=0) on X_18
db_labels_18 = ...    # c) DBSCAN(eps=0.2, min_samples=5) on X_18
scan_18 = ...         # d) {eps: summary_18(labels of DBSCAN(eps=eps, min_samples=5))} for eps in EPS_18
hdb_labels_18 = ...   # e) HDBSCAN(min_cluster_size=10) on X_18

In [ ]:
with wb.attempt("7.18"):
    toy_18 = summary_18(np.where(group_18 == 3, 2, group_18))           # the two blobs merged into one cluster
    toy_ok_18 = toy_18 is not None and np.size(toy_18) == 3 and list(toy_18)[0] == 3 and list(toy_18)[1] == 30
    if returned("7.18", "summary_18", toy_18):
        verdict("7.18", toy_ok_18,
                "summary_18 compte bien les clusters (sans le bruit) et les points de bruit.",
                "summary_18(labels) doit renvoyer (nombre de clusters sans compter -1, nombre de points -1, ARI) ; "
                f"sur un découpage test, j'attends 3 clusters et 30 points de bruit, j'obtiens {toy_18}.")
    if toy_ok_18:
        real_18 = group_18 != -1
        toy_ari_18 = adjusted_rand_score(group_18[real_18], np.where(group_18 == 3, 2, group_18)[real_18])
        toy_ok_18 = abs(float(list(toy_18)[2]) - toy_ari_18) < 1e-9
        verdict("7.18", toy_ok_18, "son ARI ne compare que les points qui ne sont pas du bruit dans group_18.",
                "l'ARI de summary_18 ne compare pas les bons points : seulement ceux qui ne sont PAS du bruit dans "
                "group_18 (le bruit des algorithmes, lui, reste compté).")
    if toy_ok_18 and not (scan_18 is ... or isinstance(scan_18, dict)):
        print("❌ Ex 7.18 : scan_18 doit être un dictionnaire {eps: summary_18(labels)}, une entrée par valeur de EPS_18.")
        toy_ok_18 = False
    if not toy_ok_18:
        pass
    elif filled(km_labels_18, db_labels_18, scan_18, hdb_labels_18):
        table_18 = pd.DataFrame({name: summary_18(labels) for name, labels in
                                 [("k-means, k=4", km_labels_18), ("DBSCAN, eps=0.2", db_labels_18),
                                  ("HDBSCAN, 10", hdb_labels_18)]},
                                index=["clusters", "noise points", "ARI"]).T
        scan_table_18 = pd.DataFrame(scan_18, index=["clusters", "noise points", "ARI"]).T
        for table in (table_18, scan_table_18):
            table[["clusters", "noise points"]] = table[["clusters", "noise points"]].astype(int)
            print(table.round(3), end="\n\n")
        best_eps_18 = max(scan_18, key=lambda eps: scan_18[eps][2])
        verdict("7.18", summary_18(hdb_labels_18)[0] == 4 and summary_18(hdb_labels_18)[2] > 0.95,
                f"HDBSCAN retrouve les quatre groupes (ARI {fr(summary_18(hdb_labels_18)[2])}).",
                "avec HDBSCAN(min_cluster_size=10), j'attends quatre clusters et un ARI au-dessus de 0,95.")
        verdict("7.18", not any(s[0] == 4 and s[2] > 0.95 for s in scan_18.values()),
                f"aucun eps de DBSCAN ne retrouve les quatre groupes (le meilleur ARI, {fr(scan_18[best_eps_18][2])}, "
                f"vient de eps = {fr(best_eps_18, 2)}).",
                "relis ton balayage : un même eps ne peut pas convenir aux deux lunes, proches l'une de l'autre, et au "
                "grand groupe épars.")
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
        for ax, (title, labels) in zip(axes, [("k-means, k = 4", km_labels_18), ("DBSCAN, eps = 0.2", db_labels_18),
                                             ("HDBSCAN, min_cluster_size = 10", hdb_labels_18)]):
            show_clusters(ax, X_18, labels, title)
        plt.show()
    else:
        print("⏳ Ex 7.18 : remplis d'abord km_labels_18, db_labels_18, scan_18 et hdb_labels_18.")

### Ex 7.19 — Distance au plus proche voisin quand la dimension grimpe 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir comment la distance au plus proche voisin et la distance moyenne évoluent avec la dimension.  
**Prérequis :** fiche §7.6 (pas encore la §7.6.1) · 0B (distance en dimension d) · Ex 2.25 · **Fil rouge :** synthétique · **Parcours :** M, C

Fais cet exercice **avant** de lire la §7.6.1 du livre et de la fiche : elle y répond. On tire 500 points uniformément dans le cube $[0, 1]^d$, pour $d$ = 1, 2, 5, 10, 20, 50 et 100, et l'on mesure deux moyennes : la distance entre deux points quelconques, et la distance de chaque point à son plus proche voisin.

Prédis, **avant** d'exécuter l'expérience :
a) `mean_grows_19` : la distance moyenne entre deux points grandit-elle avec $d$ ? (`True` ou `False`)
b) `nn_grows_19` : la distance moyenne au plus proche voisin grandit-elle avec $d$ ? (`True` ou `False`)
c) `ratio_19` : en dimension 100, le rapport (distance au plus proche voisin) / (distance moyenne) sera proche de `0.05`, `0.3` ou `0.8` ?

Dans tes notes : compare avec tes prédictions. Que dit ce rapport de la notion de « plus proche voisin » ? Lis ensuite la §7.6.1, et l'encadré ⚠️ sur la figure 7.25.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
mean_grows_19 = ...   # a) True or False
nn_grows_19 = ...     # b) True or False
ratio_19 = ...        # c) 0.05, 0.3 or 0.8

In [ ]:
if any(answer is ... or answer is None for answer in [mean_grows_19, nn_grows_19, ratio_19]):
    print("⏳ Ex 7.19 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    DIMS_19 = [1, 2, 5, 10, 20, 50, 100]
    rng_19 = np.random.default_rng(719)
    nearest_19, mean_19 = [], []
    for d in DIMS_19:
        points_19 = rng_19.random((500, d))                       # 500 points, uniform in the cube [0, 1]^d
        D_19 = pdist(points_19)                                   # the distances of the 124 750 pairs
        square_19 = np.full((500, 500), np.inf)
        square_19[np.triu_indices(500, k=1)] = D_19
        square_19 = np.minimum(square_19, square_19.T)            # symmetric, with inf on the diagonal
        nearest_19.append(square_19.min(axis=1).mean())
        mean_19.append(D_19.mean())
    print(pd.DataFrame({"d": DIMS_19, "nearest neighbour": nearest_19, "mean distance": mean_19,
                        "ratio": np.array(nearest_19) / np.array(mean_19)}).round(3).to_string(index=False))
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].plot(DIMS_19, mean_19, "o-", label="mean distance between two points")
    axes[0].plot(DIMS_19, nearest_19, "s-", label="mean distance to the nearest neighbour")
    axes[0].set(xscale="log", xlabel="dimension d", ylabel="distance")
    axes[0].legend(fontsize=8)
    axes[1].plot(DIMS_19, np.array(nearest_19) / np.array(mean_19), "o-", color="C2")
    axes[1].set(xscale="log", xlabel="dimension d", ylabel="nearest / mean", ylim=(0, 1))
    plt.show()
    print(f"your predictions: a) {mean_grows_19}   b) {nn_grows_19}   c) {ratio_19}")

### Ex 7.20 — Densité, plus proche voisin et concentration des distances 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer la densité d'échantillons et la concentration des distances quand la dimension grandit, et comparer avec des données réelles.  
**Prérequis :** Ex 7.19 · ✏️ 7.4 · Ex 7.13 · Ex 2.25 (contraste) · fiche §7.6, §7.6.1 · **Fil rouge :** synthétique/MNIST · **Parcours :** M, C

Trois mesures de la malédiction de la dimension, avec tes fonctions.

a) écris `occupied_cells_20(X, bins)` : le nombre de cases **non vides** quand chaque axe de $[0, 1]^d$ est découpé en `bins` cases égales. Le numéro de case d'une coordonnée $x$ est la partie entière de $x \times$ `bins` (`np.floor`), sans dépasser `bins - 1` ; une case est une ligne de numéros, et `np.unique(..., axis=0)` garde les lignes distinctes. La vérification tire 1 000 points dans $[0, 1]^8$ et compte les cases occupées en dimension 1 à 8, avec 4 cases par axe ; elle vérifie les valeurs en dimension 5 et 8 ;
b) `expected_20` : avec la formule de ✏️ 7.4, le nombre **moyen** de cases occupées par 1 000 points en dimension 5, avec 4 cases par axe (2 décimales) ;
c) écris `nn_and_mean_20(X)` : le couple (distance moyenne de chaque point à son plus proche voisin, distance moyenne entre deux points distincts), avec ta `pairwise_sq_distances` (7.13) et une racine carrée. L'identité laisse sur la diagonale de minuscules valeurs au lieu de 0 : mets-la à 0 (`np.fill_diagonal`), puis écarte-la des minimums et des moyennes, car un point n'est pas son propre voisin. La vérification calcule le rapport des deux en dimension 2, 10, 50 et 200, pour 500 points uniformes, et vérifie celui de la dimension 50 ;
d) écris `contrast_20(X)` : la moyenne, sur tous les points, du contraste de 2.25, $\frac{d_{\max} - d_{\min}}{d_{\min}}$ (plus grande et plus petite distance du point aux **autres** points). La vérification le compare pour 500 points uniformes de $[0, 1]^{784}$ et pour 500 images de MNIST, qui ont aussi 784 pixels.

Dans tes notes : en a), à partir de quelle dimension la plupart des cases sont-elles vides ? Que dit le rapport de c) sur un classifieur des plus proches voisins ? Pourquoi les images de MNIST gardent-elles un contraste bien plus grand que des points uniformes (fiche, bénédiction de la structure) ?

In [ ]:
def occupied_cells_20(X, bins):
    """Number of non-empty cells when each axis of [0, 1]^d is cut into `bins` equal cells."""
    raise NotImplementedError("occupied_cells_20() is not written yet")


def nn_and_mean_20(X):
    """(mean distance from each point to its nearest neighbour, mean distance between two distinct points)."""
    raise NotImplementedError("nn_and_mean_20() is not written yet")


def contrast_20(X):
    """Mean over the points of (d_max - d_min) / d_min, the contrast of Ex 2.25 seen from every point."""
    raise NotImplementedError("contrast_20() is not written yet")


expected_20 = ...   # b) the mean number of occupied cells for 1000 points, d = 5, 4 cells per axis

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.20"):
    U_20 = np.random.default_rng(720).random((1000, 8))       # 1000 points, uniform in [0, 1]^8
    cells_20 = [occupied_cells_20(U_20[:, :d], 4) for d in range(1, 9)]
    if returned("7.20", "occupied_cells_20", cells_20[0]):
        print(pd.DataFrame({"d": range(1, 9), "cells": [4 ** d for d in range(1, 9)], "occupied": cells_20,
                            "density n / 4^d": [round(1000 / 4 ** d, 4) for d in range(1, 9)]}).to_string(index=False))
        wb.check("7.20a", [cells_20[4], cells_20[7]], computed=True)
wb.check("7.20b", expected_20)
with wb.attempt("7.20"):
    rng_20 = np.random.default_rng(7201)
    pairs_20 = {d: nn_and_mean_20(rng_20.random((500, d))) for d in [2, 10, 50, 200]}
    if returned("7.20", "nn_and_mean_20", pairs_20[2]) and np.size(pairs_20[2]) != 2:
        print("❌ Ex 7.20 : nn_and_mean_20 doit renvoyer deux nombres : (plus proche voisin, distance moyenne).")
    elif pairs_20[2] is not None:
        ratios_20 = {d: pair[0] / pair[1] for d, pair in pairs_20.items()}
        print("nearest / mean:", {d: round(float(r), 3) for d, r in ratios_20.items()})
        wb.check("7.20c", ratios_20[50], computed=True)
with wb.attempt("7.20"):
    digits_20, _ = wb.datasets.load_mnist("train", n=500, flatten=True, normalize=True, seed=0)
    random_20 = np.random.default_rng(7202).random((500, 784))
    contrasts_20 = [contrast_20(random_20), contrast_20(digits_20.astype(float))]
    if returned("7.20", "contrast_20", contrasts_20[0]) and np.size(contrasts_20[0]) != 1:
        print("❌ Ex 7.20 : contrast_20 doit renvoyer un seul nombre, la moyenne des contrastes des 500 points.")
    elif contrasts_20[0] is not None:
        print(f"contrast: uniform points in [0, 1]^784 {float(contrasts_20[0]):.3f} · "
              f"MNIST {float(contrasts_20[1]):.3f}")
        wb.check("7.20d", contrasts_20, computed=True)

### Ex 7.21 — Reproduire les figures 7.27 et 7.30 (boule/cube, hyper-orange) 🎨 ★★ ⏱️ 25 min
**Objectif :** tracer les vraies valeurs de deux figures du livre et les confirmer par une simulation.  
**Prérequis :** ∂ 7.5 · ∂ 7.6 · fiche §7.6.1 (encadrés 🧮 et ⚠️ sur la figure 7.27) · **Fil rouge :** synthétique · **Parcours :** M, C

a) écris `ball_ratio_21(d_max)`, qui renvoie le tableau des rapports $q_d = \frac{V_d}{2^d}$ pour $d = 1, \ldots, d_{\max}$, par la récurrence de ∂ 7.6 ($V_1 = 2$, $V_2 = \pi$, $V_d = \frac{2\pi}{d} V_{d-2}$) ;
b) écris `orange_radius_21(d)`, le rayon de l'hyper-orange de ∂ 7.5, pour un nombre ou un tableau de dimensions.

La vérification trace les deux figures, et ajoute à la première une estimation de **Monte-Carlo** : la part de 200 000 points tirés uniformément dans le cube $[-1, 1]^d$ qui tombent dans la boule de rayon 1 (méthode de 3.12). Elle vérifie ensuite tes valeurs.

Dans tes notes : compare tes courbes aux figures 7.27 et 7.30 du livre (encadré ⚠️ de la fiche). Combien des 200 000 points tomberaient en moyenne dans la boule en dimension 15, puis 20 ? Que vaudrait alors l'estimation de Monte-Carlo ?

In [ ]:
def ball_ratio_21(d_max):
    """The ratios V_d / 2^d for d = 1, ..., d_max (an array of length d_max)."""
    raise NotImplementedError("ball_ratio_21() is not written yet")


def orange_radius_21(d):
    """The radius of the hyper-orange in dimension d (a number or an array of dimensions)."""
    raise NotImplementedError("orange_radius_21() is not written yet")

In [ ]:
with wb.attempt("7.21"):
    dims_21 = np.arange(1, 21)
    q_21, r_21 = ball_ratio_21(20), orange_radius_21(dims_21)
    if returned("7.21", "ball_ratio_21", q_21) and returned("7.21", "orange_radius_21", r_21):
        q_21, r_21 = np.asarray(q_21, dtype=float), np.asarray(r_21, dtype=float)
        exact_21 = [np.pi / 4, np.pi / 6, np.pi ** 2 / 32]
        verdict("7.21", q_21.shape == (20,) and np.allclose(q_21[1:4], exact_21) and abs(q_21[0] - 1) < 1e-12,
                "ball_ratio_21 donne les valeurs exactes (1, π/4, π/6, π²/32…).",
                "ball_ratio_21(20) doit renvoyer 20 rapports V_d / 2^d, de d = 1 à 20 (1, puis π/4 ≈ 0,785…).")
        verdict("7.21", r_21.shape == (20,) and np.allclose(r_21, np.sqrt(dims_21) - 1),
                "orange_radius_21 est juste pour d = 1 à 20.",
                "orange_radius_21(d) doit valoir √d − 1, pour un tableau de dimensions aussi.")
        if q_21.shape == (20,) and r_21.shape == (20,):
            rng_21 = np.random.default_rng(721)
            monte_carlo_21 = [np.mean((rng_21.uniform(-1, 1, (200_000, d)) ** 2).sum(axis=1) <= 1) for d in range(1, 11)]
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].semilogy(dims_21, q_21, "o-", label="exact: V_d / 2^d")
            axes[0].semilogy(range(1, 11), monte_carlo_21, "x", ms=10, color="C3", label="Monte Carlo (200 000 points)")
            axes[0].set(xlabel="dimension d", ylabel="ball volume / cube volume", xticks=range(0, 21, 2),
                        title="The ball in the cube (fig. 7.27)")
            axes[0].legend(fontsize=8)
            axes[1].plot(dims_21, r_21, "o-", label="radius of the orange: sqrt(d) - 1")
            axes[1].axhline(1, color="C1", ls="--", label="radius of a balloon")
            axes[1].axhline(2, color="C2", ls=":", lw=2, label="half-width of the box")
            axes[1].set(xlabel="dimension d", ylabel="radius", xticks=range(0, 21, 2), title="The hyper-orange (fig. 7.30)")
            axes[1].legend(fontsize=8)
            plt.show()
            verdict("7.21", max(abs(m - q) for m, q in zip(monte_carlo_21, q_21[:10])) < 0.01,
                    "les points de Monte-Carlo tombent sur la courbe exacte.",
                    "les points de Monte-Carlo s'écartent de ta courbe : relis la récurrence de ∂ 7.6.")

## Partie C · Un-contre-tous et un-contre-un génériques dans mylearn.multiclass

Fiche §7.4. Tu écris `mylearn.multiclass` : deux **méta-estimateurs**, qui transforment n'importe quel classifieur binaire en classifieur multi-classe, puis tu compares les trois stratégies sur des données réelles. Ces classes resserviront avec le perceptron (ch. 10) et les SVM (ch. 13). Les exercices réutilisent les quatre classes de 7.16 (`X_16`, `y_16`, `X_test_16`, `y_test_16`) : exécute d'abord les cellules de données des parties A et B. La cellule ci-dessous prépare l'exemple des docstrings.

In [ ]:
X_DOC = np.array([[0.0, 0.0], [0.0, 1.0], [5.0, 0.0], [5.0, 1.0], [0.0, 5.0], [1.0, 5.0]])   # the docstring examples
Y_DOC = np.array([0, 0, 1, 1, 2, 2])
X_NEW_DOC = np.array([[0.5, 0.5], [5.0, 0.5], [0.5, 5.0]])
print("the four classes of Ex 7.16:", X_16.shape, "training points,", X_test_16.shape, "test points")

### Ex 7.22 — Un-contre-tous générique : OneVsRestClassifier 🔨 ★★★ ⏱️ 40 min
**Objectif :** écrire un méta-estimateur qui entraîne une copie d'un classifieur binaire par classe.  
**Prérequis :** Ex 7.14 · ✏️ 7.1 · 0A (classes, `copy.deepcopy`) · fiche §7.4.1 · **Fil rouge :** synthétique · **mylearn :** `multiclass.py` · **Parcours :** C

> **mylearn** : ouvre `mon_travail/mylearn/multiclass.py` (créé par `python tools/start_chapter.py 7`). Mêmes règles qu'en 7.13 : NumPy et le module `copy` de Python sont permis ; les classes de `sklearn.multiclass` sont les **oracles** des tests. Enregistre, puis relance la cellule de vérification.

Écris la classe `OneVsRestClassifier` (lis ses docstrings). C'est un **méta-estimateur** : il reçoit un classifieur binaire **non entraîné**, `estimator`, et en entraîne une copie par classe.
- `fit(X, y)` : `classes_` est le tableau trié des labels distincts (`np.unique`). Lève une `ValueError` s'il y a moins de 3 classes (avec deux classes, on utilise directement le classifieur binaire), ou si `estimator` n'a ni `decision_function` ni `predict_proba` (`hasattr`). Puis, pour chaque classe `c` de `classes_`, une copie `copy.deepcopy(self.estimator)` apprend les labels `(y == c).astype(int)` : 1 pour la classe `c`, 0 pour toutes les autres. `estimators_` garde ces modèles dans l'ordre de `classes_`. `estimator` lui-même n'est jamais entraîné ;
- `decision_function(X)` : une colonne par modèle, son `decision_function(X)`, ou la colonne 1 de son `predict_proba(X)` s'il n'a pas de `decision_function` ;
- `predict(X)` : la classe de la plus grande colonne (`np.argmax` garde la première en cas d'égalité) ;
- `score(X, y)` : l'accuracy, un `float`.

La vérification reprend l'exemple de la docstring, puis les quatre classes de 7.16 :
a) l'accuracy de ton un-contre-tous, autour de ton `NearestCentroid`, sur les points de test ;
elle vérifie ensuite qu'il donne les mêmes prédictions que le calcul fait à la main en 7.16, et lance les tests, qui comparent ta classe à `sklearn.multiclass.OneVsRestClassifier` autour d'autres classifieurs binaires de scikit-learn (`LinearSVC`, `LogisticRegression`, `GaussianNB` : tu les rencontreras au ch. 13).

Dans tes notes : pourquoi copier l'estimateur ? Que donnerait `self.estimators_.append(self.estimator.fit(X, labels))` à chaque tour de boucle (indice : `fit` renvoie `self`) ? Pourquoi faut-il un score, et pas seulement `predict` ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.22"):
    doc_22 = fitted(mylearn.multiclass.OneVsRestClassifier(mylearn.cluster.NearestCentroid()), X_DOC, Y_DOC)
    print("docstring example:", len(doc_22.estimators_), "models · decision_function",
          np.asarray(doc_22.decision_function(X_NEW_DOC)).tolist(), "· predict",
          np.asarray(doc_22.predict(X_NEW_DOC)).tolist())
    ovr_22 = fitted(mylearn.multiclass.OneVsRestClassifier(mylearn.cluster.NearestCentroid()), X_16, y_16)
    score_22 = ovr_22.score(X_test_16, y_test_16)
    if returned("7.22", "score", score_22):
        wb.check("7.22a", score_22, computed=True)
    classes_22 = np.unique(y_16)                                 # the one-versus-rest of Ex 7.16, by hand
    binary_22 = [fitted(mylearn.cluster.NearestCentroid(), X_16, (y_16 == c).astype(int)) for c in classes_22]
    by_hand_22 = classes_22[np.column_stack([model.decision_function(X_test_16) for model in binary_22]).argmax(axis=1)]
    verdict("7.22", np.array_equal(np.asarray(ovr_22.predict(X_test_16)), by_hand_22),
            "les mêmes prédictions que l'un-contre-tous fait à la main en 7.16.",
            "tes prédictions diffèrent de l'un-contre-tous fait à la main en 7.16 : relis fit et predict.")
    run_mylearn_tests("multiclass", "test_one_vs_rest_")

### Ex 7.23 — Un-contre-un générique : OneVsOneClassifier 🔨 ★★★ ⏱️ 45 min
**Objectif :** écrire le méta-estimateur des duels et de leurs votes, et comprendre ce qu'il fait des centroïdes.  
**Prérequis :** Ex 7.22 · ✏️ 7.2 · fiche §7.4.2 · **Fil rouge :** synthétique · **mylearn :** `multiclass.py` · **Parcours :** C

> **mylearn** : toujours dans `mon_travail/mylearn/multiclass.py`, mêmes règles qu'en 7.22.

Écris la classe `OneVsOneClassifier` (lis ses docstrings) :
- `fit(X, y)` : `classes_` comme en 7.22 (une `ValueError` s'il y a moins de 3 classes) ; `pairs_` est la liste des couples d'**indices** de classes `(i, j)` avec `i < j`, dans l'ordre (0, 1), (0, 2)… (1, 2)… ; pour chaque couple, une copie de l'estimateur apprend sur les **seuls** échantillons des classes `classes_[i]` et `classes_[j]` (dans l'ordre que tu veux), avec le label 1 pour la classe `j` et 0 pour la classe `i` ; `estimators_[m]` est le modèle du couple `pairs_[m]` ;
- `votes(X)` : un tableau d'**entiers** de forme (n, K) ; chaque modèle donne une voix à la classe `j` de son couple quand son `predict` vaut 1, à la classe `i` sinon. Chaque ligne totalise donc $\frac{K(K-1)}{2}$ voix ;
- `predict(X)` : la classe qui a le plus de voix (`np.argmax` : en cas d'égalité, le plus petit indice) ;
- `score(X, y)` : l'accuracy.

La vérification reprend l'exemple de la docstring, puis les quatre classes de 7.16 :
a) les voix des cinq premiers points de test (`votes`, un tableau de forme (5, 4)) ;
b) l'accuracy de ton un-contre-un sur les points de test ;
elle compare ensuite tes prédictions à celles du centroïde le plus proche multi-classe, dessine les régions de l'un-contre-tous et de l'un-contre-un, et lance les tests.

Dans tes notes : avec des centroïdes, l'un-contre-un donne **exactement** les prédictions du centroïde le plus proche multi-classe. Pourquoi ? Combien de voix reçoit toujours la classe prédite ? Pourquoi la classe sacrifiée en 7.16 ne l'est-elle plus ? Dans `votes`, pourquoi parler d'indices de classes plutôt que de labels ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.23"):
    doc_23 = fitted(mylearn.multiclass.OneVsOneClassifier(mylearn.cluster.NearestCentroid()), X_DOC, Y_DOC)
    print("docstring example: pairs_", doc_23.pairs_, "· votes", np.asarray(doc_23.votes(X_NEW_DOC)).tolist(),
          "· predict", np.asarray(doc_23.predict(X_NEW_DOC)).tolist())
    ovo_23 = fitted(mylearn.multiclass.OneVsOneClassifier(mylearn.cluster.NearestCentroid()), X_16, y_16)
    print("pairs_ of the four classes:", ovo_23.pairs_)
    votes_23 = ovo_23.votes(X_test_16[:5])
    if returned("7.23", "votes", votes_23):
        print("votes of the first five test points (columns A, B, C, D):\n", np.asarray(votes_23))
        wb.check("7.23a", votes_23, computed=True)
    score_23 = ovo_23.score(X_test_16, y_test_16)
    if returned("7.23", "score", score_23):
        wb.check("7.23b", score_23, computed=True)
    native_23 = fitted(mylearn.cluster.NearestCentroid(), X_16, y_16)
    same_23 = np.array_equal(np.asarray(ovo_23.predict(X_test_16)), np.asarray(native_23.predict(X_test_16)))
    verdict("7.23", same_23,
            "l'un-contre-un donne exactement les prédictions du centroïde le plus proche multi-classe.",
            "tes prédictions diffèrent de celles du centroïde le plus proche multi-classe : relis votes et predict.")
    if same_23:
        ovr_23 = fitted(mylearn.multiclass.OneVsRestClassifier(mylearn.cluster.NearestCentroid()), X_16, y_16)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
        for ax, (title, model) in zip(axes, [("one-versus-rest (7.22)", ovr_23), ("one-versus-one (7.23)", ovo_23)]):
            wb.plot.plot_decision_boundary(model, X_test_16, y_test_16, ax=ax, class_names=list(np.unique(y_16)))
            ax.set_title(title)
        plt.show()
    run_mylearn_tests("multiclass", "test_one_vs_one_")

### Ex 7.24 — OvR, OvO ou multi-classe natif : accuracy, nombre de modèles, temps 🔬 ★★★ ⏱️ 35 min
**Objectif :** comparer les trois stratégies sur deux jeux réels, et relier le coût d'entraînement à la taille des données.  
**Prérequis :** Ex 7.22 · Ex 7.23 · ✏️ 7.1 · ch. 1 (jeux d'entraînement et de test) · fiche §7.3 et §7.4 · **Fil rouge :** Penguins/MNIST · **Parcours :** C

On compare trois façons de traiter plusieurs classes avec **ton** `NearestCentroid` : le modèle multi-classe natif, l'un-contre-tous (7.22) et l'un-contre-un (7.23). Deux jeux réels : les manchots (3 espèces, 4 mesures ; 233 pour l'entraînement, `TRAIN_PENG`, et 100 pour le test, `TEST_PENG`, le découpage du ch. 1) et MNIST (10 chiffres, 784 pixels ; 10 000 images pour l'entraînement, 2 000 pour le test).

a) `Z_train_24`, `Z_test_24` : les mesures des manchots d'entraînement et de test, standardisées avec la moyenne et l'écart-type (`ddof=0`) des manchots d'**entraînement** seulement. Le jeu de test ne doit rien apprendre au modèle, pas même sa moyenne (ch. 1, et le ch. 8 y reviendra) ;
b) écris `compare_24(X_train, y_train, X_test, y_test)`, qui renvoie un `DataFrame` de trois lignes, dont l'**index** (le nom des lignes) est `"native"`, `"OvR"` et `"OvO"`, et quatre colonnes : `n_models` (le nombre de modèles entraînés : 1 pour le modèle natif, `len(model.estimators_)` pour les deux autres), `accuracy` (sur le jeu de test), `fit_s` et `predict_s` (les durées de `fit` et de `predict`, en secondes, mesurées avec `time.perf_counter()`).

La vérification lance `compare_24` sur les manchots et sur MNIST, puis affiche le recall de chaque espèce de manchot avec les trois stratégies.

Dans tes notes : combien de modèles pour $K$ classes avec chaque stratégie (✏️ 7.1) ? Sur MNIST, l'un-contre-un entraîne 45 modèles et l'un-contre-tous 10 : pourquoi l'un-contre-un ne met-il pas 4,5 fois plus de temps à s'entraîner ? (compte les lignes que voit chaque modèle) Qu'est-ce qui coûte, à la prédiction ? Quelle espèce l'un-contre-tous sacrifie-t-il, et pourquoi (7.16) ? Quand préférer l'un-contre-un (fiche §7.4.2) ?

In [ ]:
X_mnist_24, y_mnist_24 = wb.datasets.load_mnist("train", n=10_000, flatten=True, normalize=True, seed=0)
X_mnist_test_24, y_mnist_test_24 = wb.datasets.load_mnist("test", n=2_000, flatten=True, normalize=True, seed=0)


def table_problem_24(table):
    """'' when compare_24 returned the DataFrame of the statement, otherwise what is wrong with it."""
    if table is None:
        return "compare_24 renvoie None : as-tu oublié le return ?"
    if not isinstance(table, pd.DataFrame):
        return f"compare_24 doit renvoyer un DataFrame, pas un objet de type {type(table).__name__}."
    if not {"native", "OvR", "OvO"} <= set(table.index):
        return f"l'index (les noms des lignes) doit être native, OvR, OvO ; j'obtiens {list(table.index)}."
    if not {"n_models", "accuracy", "fit_s", "predict_s"} <= set(table.columns):
        return f"les colonnes doivent être n_models, accuracy, fit_s, predict_s ; j'obtiens {list(table.columns)}."
    return ""

print("MNIST:", X_mnist_24.shape, X_mnist_test_24.shape, "· penguins:", len(TRAIN_PENG), "for training,",
      len(TEST_PENG), "for the test")

In [ ]:
Z_train_24 = ...   # a) X_peng[TRAIN_PENG] standardised with the mean and std of the TRAINING penguins
Z_test_24 = ...    # a) X_peng[TEST_PENG] standardised with the SAME mean and std


def compare_24(X_train, y_train, X_test, y_test):
    """DataFrame, one row per strategy ("native", "OvR", "OvO"), columns n_models, accuracy, fit_s, predict_s."""
    raise NotImplementedError("compare_24() is not written yet")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.24"):
    tables_24 = {}
    if filled(Z_train_24, Z_test_24):
        mean_24, std_24 = X_peng[TRAIN_PENG].mean(axis=0), X_peng[TRAIN_PENG].std(axis=0)
        verdict("7.24", np.shape(Z_train_24) == (len(TRAIN_PENG), 4) and np.shape(Z_test_24) == (len(TEST_PENG), 4)
                and np.allclose(Z_train_24, (X_peng[TRAIN_PENG] - mean_24) / std_24)
                and np.allclose(Z_test_24, (X_peng[TEST_PENG] - mean_24) / std_24),
                "a) les deux tableaux sont standardisés avec la moyenne et l'écart-type des manchots d'entraînement.",
                "a) standardise X_peng[TRAIN_PENG] et X_peng[TEST_PENG] avec la moyenne et l'écart-type (ddof=0) des "
                "manchots d'ENTRAÎNEMENT seulement.")
        tables_24["penguins"] = compare_24(Z_train_24, species[TRAIN_PENG], Z_test_24, species[TEST_PENG])
    else:
        print("⏳ Ex 7.24 : remplis Z_train_24 et Z_test_24 (question a) pour comparer aussi les manchots.")
    tables_24["MNIST"] = compare_24(X_mnist_24, y_mnist_24, X_mnist_test_24, y_mnist_test_24)
    problems_24 = [table_problem_24(table) for table in tables_24.values()]
    if any(problems_24):
        print("❌ Ex 7.24 :", next(problem for problem in problems_24 if problem))
    else:
        for name, table in tables_24.items():
            print(f"--- {name}\n{table.round(4)}")
        expected_24 = {"penguins": [1, 3, 3], "MNIST": [1, 10, 45]}
        verdict("7.24", all(list(t.loc[["native", "OvR", "OvO"], "n_models"]) == expected_24[name]
                            for name, t in tables_24.items()),
                "le bon nombre de modèles : 1, K et K(K - 1)/2.",
                "n_models doit valoir 1 pour le modèle natif, len(model.estimators_) pour les deux autres.")
        verdict("7.24", all(t.loc["OvO", "accuracy"] == t.loc["native", "accuracy"] for t in tables_24.values()),
                "l'un-contre-un a exactement l'accuracy du modèle natif (7.23).",
                "l'un-contre-un avec des centroïdes devrait avoir l'accuracy du modèle natif (7.23) : relis compare_24, "
                "puis ton OneVsOneClassifier.")
    if "penguins" in tables_24:
        recalls_24 = {}
        for name, model in [("native", mylearn.cluster.NearestCentroid()),
                            ("OvR", mylearn.multiclass.OneVsRestClassifier(mylearn.cluster.NearestCentroid())),
                            ("OvO", mylearn.multiclass.OneVsOneClassifier(mylearn.cluster.NearestCentroid()))]:
            predicted = np.asarray(fitted(model, Z_train_24, species[TRAIN_PENG]).predict(Z_test_24))
            recalls_24[name] = {s: np.mean(predicted[species[TEST_PENG] == s] == s) for s in np.unique(species)}
        print("--- recall of each species of penguin (100 test penguins)\n", pd.DataFrame(recalls_24).T.round(3))

## Partie D · k-means de zéro, silhouette, choix de k, phénomène de Hughes, défi

Fiche §7.5 (Lloyd, k-means++, « Combien de clusters ? ») et §7.6. Tu écris k-means dans `mylearn.cluster` : l'initialisation k-means++, puis la classe `KMeans` ; tu débusques les bugs d'un k-means écrit par un collègue, tu programmes la silhouette, tu choisis $k$ sans labels, tu mesures le phénomène de Hughes, et tu relèves le défi des manchots. La cellule ci-dessous standardise les 333 manchots de la partie B.

In [ ]:
Z_peng = (X_peng - X_peng.mean(axis=0)) / X_peng.std(axis=0)   # the 333 penguins, standardised (as in 7.17 c)
years = penguins["year"].to_numpy()
print(Z_peng.shape, {int(year): int(count) for year, count in zip(*np.unique(years, return_counts=True))})

### Ex 7.25 — Initialisation k-means++ 🔨 ★★★ ⏱️ 35 min
**Objectif :** programmer le tirage pondéré de k-means++ et mesurer ce qu'il apporte face à des centres tirés au hasard.  
**Prérequis :** Ex 7.13 · ✏️ 7.3 · ch. 2 (générateur aléatoire) · fiche §7.5 (encadré 🧮 sur k-means++) · **Fil rouge :** synthétique · **mylearn :** `cluster.py` · **Parcours :** R, C

> **mylearn** : dans `mon_travail/mylearn/cluster.py`, mêmes règles qu'en 7.13 (NumPy permis, SciPy et scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris `kmeans_plusplus(X, n_clusters, rng=None)` (lis sa docstring), la version d'origine de k-means++ (fiche, encadré 🧮) :
- convertis `X` en tableau de flottants à deux dimensions ; lève une `ValueError` si `n_clusters` n'est pas entre 1 et le nombre de lignes **distinctes** de `X` (`len(np.unique(X, axis=0))`) ; si `rng` vaut `None`, prends `np.random.default_rng()` ;
- le premier centre est une ligne tirée uniformément (`rng.integers(len(X))`) ;
- garde à jour le tableau des $D(\mathbf{x})^2$, le carré de la distance de chaque point au centre déjà choisi le plus proche : après chaque nouveau centre, `np.minimum` avec les carrés des distances à ce centre ;
- chaque centre suivant est la ligne d'indice `rng.choice(len(X), p=d2 / d2.sum())` ;
- renvoie les lignes choisies, dans l'ordre du tirage : `X[indices]`, avec la liste des indices tirés (l'indexation par une liste crée une copie).

La vérification tire 300 fois six centres sur les six paquets ci-dessous, avec ton k-means++ et avec six lignes tirées au hasard. Elle compte les paquets qui reçoivent au moins un centre, puis lance l'algorithme de Lloyd (le `KMeans` de scikit-learn, depuis ces centres) et compte les départs qui mènent au meilleur découpage connu. Elle lance enfin les tests, qui vérifient tes tirages par un test du khi-deux.

Dans tes notes : pourquoi des lignes tirées au hasard laissent-elles souvent des paquets sans centre ? Que fait l'algorithme de Lloyd d'un paquet sans centre de départ ? Pourquoi un point déjà choisi ne peut-il pas être tiré une seconde fois ?

In [ ]:
BLOB_CENTRES_25 = np.array([[0.0, 0.0], [6.0, 0.0], [12.0, 0.0], [0.0, 6.0], [6.0, 6.0], [12.0, 6.0]])
BLOB_SIZES_25 = [150, 30, 30, 30, 30, 30]                     # one big blob and five small ones
rng_25 = np.random.default_rng(725)
X_25 = np.vstack([rng_25.normal(centre, 0.6, (size, 2)) for centre, size in zip(BLOB_CENTRES_25, BLOB_SIZES_25)])
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.scatter(X_25[:, 0], X_25[:, 1], s=8, color="gray")
ax.set(title="Six blobs: 150 points in the first one, 30 in each of the others", aspect="equal")
plt.show()

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.25"):
    X_doc_25 = np.array([[0.0, 0.0], [0.1, 0.0], [10.0, 0.0], [10.1, 0.0]])
    doc_25 = mylearn.cluster.kmeans_plusplus(X_doc_25, 2, rng=np.random.default_rng(0))
    if returned("7.25", "kmeans_plusplus", doc_25) and np.shape(doc_25) != (2, 2):
        print(f"❌ Ex 7.25 : kmeans_plusplus doit renvoyer les lignes de X choisies, un tableau de forme "
              f"(n_clusters, n_features) ; sur l'exemple de la docstring, j'attends (2, 2), j'obtiens {np.shape(doc_25)}.")
    elif doc_25 is not None:
        print("docstring example:", np.asarray(doc_25).tolist())
        best_25 = SklearnKMeans(n_clusters=6, n_init=50, random_state=0).fit(X_25).inertia_   # the best partition known
        rng_draws_25 = np.random.default_rng(7250)
        covered_25 = {"k-means++": [], "random rows": []}
        best_found_25 = {"k-means++": [], "random rows": []}
        with threadpool_limits(limits=1):                  # 600 tiny fits: one thread each is faster
            for _ in range(300):
                starts = {"k-means++": np.asarray(mylearn.cluster.kmeans_plusplus(X_25, 6, rng=rng_draws_25), dtype=float),
                          "random rows": X_25[rng_draws_25.choice(len(X_25), size=6, replace=False)]}
                for name, centres in starts.items():
                    blobs = ((centres[:, None, :] - BLOB_CENTRES_25[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
                    covered_25[name].append(len(np.unique(blobs)))
                    lloyd = SklearnKMeans(n_clusters=6, init=centres, n_init=1).fit(X_25)   # Lloyd from this start
                    best_found_25[name].append(lloyd.inertia_ <= best_25 * 1.001)
        table_25 = pd.DataFrame({name: {"blobs covered (mean)": np.mean(covered_25[name]),
                                        "all 6 blobs covered": np.mean(np.array(covered_25[name]) == 6),
                                        "best partition after Lloyd": np.mean(best_found_25[name])}
                                 for name in covered_25}).T
        print(table_25.round(3))
        verdict("7.25", table_25.loc["k-means++", "blobs covered (mean)"]
                >= table_25.loc["random rows", "blobs covered (mean)"] + 1,
                "k-means++ couvre en moyenne bien plus de paquets que des points tirés au hasard.",
                "ton k-means++ ne couvre pas plus de paquets que le hasard : chaque nouveau centre doit être tiré avec "
                "une probabilité proportionnelle à D(x)², le carré de la distance au centre déjà choisi le plus proche.")
        fig, ax = plt.subplots(figsize=(6.5, 3.6))
        ax.hist([covered_25["random rows"], covered_25["k-means++"]], bins=np.arange(0.5, 7.5),
                label=["random rows", "k-means++"])
        ax.set(xlabel="number of blobs that receive an initial centre", ylabel="draws (out of 300)")
        ax.legend()
        plt.show()
    run_mylearn_tests("cluster", "test_kmeans_plusplus_")

### Ex 7.26 — k-means de Lloyd : la classe KMeans 🔨 ★★★ ⏱️ 60 min
**Objectif :** écrire un k-means complet : plusieurs départs, l'algorithme de Lloyd et sa règle d'arrêt, les attributs appris.  
**Prérequis :** ✏️ 7.3 · Ex 7.13 · Ex 7.25 · fiche §7.5 (encadré 🧮 sur l'algorithme de Lloyd) · **Fil rouge :** synthétique · **mylearn :** `cluster.py` · **Parcours :** R, C

> **mylearn** : dans `mon_travail/mylearn/cluster.py`, mêmes règles qu'en 7.13 (NumPy permis, SciPy et scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris la classe `KMeans` (lis ses docstrings ; `__init__` est fourni). Le plus long est `fit(X, y=None)` :
- convertis `X` (deux dimensions, flottants) et valide les hyperparamètres : `ValueError` si `n_clusters` n'est pas un entier entre 1 et le nombre de lignes, si `n_init < 1`, si `max_iter < 1`, si `init` est une chaîne autre que `"k-means++"` et `"random"`, ou un tableau qui n'a pas la forme `(n_clusters, n_features)` ;
- crée **un** générateur, `rng = np.random.default_rng(self.random_state)`, utilisé par toutes les initialisations, et la tolérance absolue `tol = self.tol * np.mean(np.var(X, axis=0))` (la règle de scikit-learn) ;
- fais `n_init` **exécutions** (une seule si `init` est un tableau). Chacune part de centres initiaux : `kmeans_plusplus(X, k, rng=rng)` (7.25) pour `"k-means++"`, `X[rng.choice(n, size=k, replace=False)]` pour `"random"`, une **copie** du tableau sinon ;
- une exécution répète, au plus `max_iter` fois : (1) l'**affectation** de chaque point au centre le plus proche (`pairwise_sq_distances`) ; (2) la **mise à jour** de chaque centre, moyenne de ses points (un centre que personne n'a rejoint ne bouge pas), et le déplacement total des centres, la somme des carrés de leurs déplacements ; puis elle s'arrête si l'affectation est identique à celle de l'itération précédente, ou si le déplacement total est inférieur ou égal à `tol`. `n_iter_` est le nombre d'itérations faites, la dernière comprise. Si l'arrêt ne vient pas d'une affectation inchangée, recalcule l'affectation avec les centres finaux ;
- l'inertie d'une exécution est la somme des carrés des distances de chaque point à son centre ; garde l'exécution d'inertie **strictement** plus faible (la première en cas d'égalité), et ses `cluster_centers_`, `labels_`, `inertia_` (un `float`) et `n_iter_` (un `int`). `fit` renvoie `self`.

Puis `predict` (le centre le plus proche), `fit_predict` (une ligne), `transform` (les distances, **sans** carré, à chaque centre) et `score` (moins l'inertie de `X` par rapport aux centres appris).

La vérification reprend l'exemple de la docstring, puis lance ton `KMeans` sur les manchots standardisés, à partir du premier Adélie, du premier Chinstrap et du premier Gentoo du tableau. Elle vérifie :
a) l'inertie finale ;
b) le nombre d'itérations ;
puis compare ton résultat au `KMeans` de scikit-learn (même départ, `n_init=1`, `algorithm="lloyd"`), relance ton `KMeans` avec k-means++ et 10 exécutions, et lance les tests.

Dans tes notes : pourquoi l'inertie ne peut-elle pas augmenter d'une itération à l'autre (fiche, encadré 🧮) ? Pourquoi un **seul** générateur pour toutes les exécutions, et pas `np.random.default_rng(self.random_state)` à chaque exécution ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.26"):
    X_doc_26 = np.array([[0.0, 0.0], [0.0, 1.0], [5.0, 5.0], [5.0, 6.0]])
    doc_26 = fitted(mylearn.cluster.KMeans(n_clusters=2, init=np.array([[0.0, 0.0], [5.0, 5.0]])), X_doc_26)
    print("docstring example: centres", np.asarray(doc_26.cluster_centers_).tolist(), "· labels",
          np.asarray(doc_26.labels_).tolist(), "· inertia", doc_26.inertia_, "· predict",
          np.asarray(doc_26.predict(np.array([[1.0, 1.0], [4.0, 4.0]]))).tolist(), "· score", doc_26.score(X_doc_26))
    INIT_26 = Z_peng[[0, 265, 146]]                           # the first Adelie, Chinstrap and Gentoo of the table
    km_26 = fitted(mylearn.cluster.KMeans(n_clusters=3, init=INIT_26), Z_peng)
    wb.check("7.26a", km_26.inertia_, computed=True)
    wb.check("7.26b", km_26.n_iter_, computed=True)
    sk_26 = SklearnKMeans(n_clusters=3, init=INIT_26, n_init=1, algorithm="lloyd").fit(Z_peng)
    centres_26 = np.asarray(km_26.cluster_centers_)
    verdict("7.26", np.array_equal(np.asarray(km_26.labels_), sk_26.labels_) and centres_26.shape == (3, 4)
            and np.allclose(centres_26, sk_26.cluster_centers_) and km_26.n_iter_ == sk_26.n_iter_,
            f"les centres, les labels et le nombre d'itérations ({km_26.n_iter_}) de scikit-learn.",
            f"scikit-learn trouve d'autres centres ou d'autres labels, ou s'arrête après {sk_26.n_iter_} itérations "
            f"(toi : {km_26.n_iter_}) : relis la règle d'arrêt.")
    km10_26 = fitted(mylearn.cluster.KMeans(n_clusters=3, random_state=0), Z_peng)   # k-means++ and 10 runs
    print("with k-means++ and 10 runs: inertia", round(float(km10_26.inertia_), 4))
    print(pd.crosstab(np.asarray(km10_26.labels_), species, rownames=["cluster"], colnames=["species"]))
    run_mylearn_tests("cluster", "test_kmeans_ and not plusplus")

### Ex 7.27 — k-means piégé : quatre bugs à débusquer 🐛 ★★★ ⏱️ 30 min
**Objectif :** trouver et corriger quatre bugs classiques d'un k-means, en les isolant l'un après l'autre.  
**Prérequis :** Ex 7.26 · ✏️ 7.3 · 0A (copies et vues, réductions par axe) · **Fil rouge :** synthétique · **Parcours :** C

Un collègue a écrit son propre k-means, `kmeans_colleague_27` (cellule ci-dessous). Il contient **quatre bugs**. La cellule le lance sur les manchots standardisés, depuis les centres initiaux de 7.26, et affiche son résultat à côté de celui de scikit-learn.

Écris `kmeans_fixed_27(X, init, max_iter=100)` : la fonction du collègue, corrigée en changeant le moins de lignes possible. Elle renvoie `(centres, labels, inertia, n_iter)` et ne doit jamais modifier `init`.

La vérification isole chaque bug : elle contrôle que `init` n'est pas modifié ; elle compare tes centres après **une** itération (`max_iter=1`) aux moyennes attendues ; elle lance ta fonction sur l'exemple de la fiche (§7.5 : les points 0, 2, 3, 9, 11, 12, centres de départ 0 et 3) ; elle recalcule l'inertie avec tes centres et tes labels. Puis elle compare ta version à scikit-learn.

Dans tes notes : pour chaque bug, le symptôme qui l'a trahi, et la correction. Pourquoi le bug de la moyenne ne se voit-il pas sur l'exemple de la fiche, en dimension 1 ?

In [ ]:
def kmeans_colleague_27(X, init, max_iter=100):
    """Lloyd's algorithm from the given initial centres: (centres, labels, inertia, n_iter)."""
    X = np.asarray(X, dtype=float)
    centres = init
    old_labels = None
    for n_iter in range(1, max_iter + 1):
        labels = ((X[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)   # (1) assignment
        for j in range(len(centres)):                                                     # (2) update
            if np.any(labels == j):
                centres[j] = X[labels == j].mean()
        old_labels = labels
        if np.array_equal(labels, old_labels):                    # the assignment did not change: converged
            break
    inertia = np.sqrt(((X - centres[labels]) ** 2).sum(axis=1)).sum()
    return centres, labels, inertia, n_iter


init_27 = Z_peng[[0, 265, 146]].copy()                         # the initial centres of Ex 7.26
start_27 = init_27.copy()                                      # kept aside, for scikit-learn
centres_c27, labels_c27, inertia_c27, n_iter_c27 = kmeans_colleague_27(Z_peng, init_27)
sk_27 = SklearnKMeans(n_clusters=3, init=start_27, n_init=1, algorithm="lloyd", tol=0).fit(Z_peng)
print("colleague:    inertia", round(float(inertia_c27), 3), "· n_iter", n_iter_c27, "· centres\n", centres_c27.round(3))
print("scikit-learn: inertia", round(sk_27.inertia_, 3), "· n_iter", sk_27.n_iter_, "· centres\n", sk_27.cluster_centers_.round(3))

In [ ]:
def kmeans_fixed_27(X, init, max_iter=100):
    """The colleague's function, corrected: (centres, labels, inertia, n_iter); init is never modified."""
    raise NotImplementedError("kmeans_fixed_27() is not written yet")

In [ ]:
with wb.attempt("7.27"):
    received_27 = Z_peng[[0, 265, 146]].copy()
    kept_27 = received_27.copy()
    result_27 = kmeans_fixed_27(Z_peng, received_27)
    if returned("7.27", "kmeans_fixed_27", result_27) and (not isinstance(result_27, (tuple, list))
                                                          or len(result_27) != 4):
        print("❌ Ex 7.27 : kmeans_fixed_27 doit renvoyer quatre valeurs : (centres, labels, inertia, n_iter).")
        result_27 = None
    if result_27 is not None:
        centres_27, labels_27, inertia_27, n_iter_27 = result_27
        verdict("7.27", np.array_equal(received_27, kept_27),
                "bug 1 corrigé : les centres initiaux reçus (init) ne sont plus modifiés.",
                "bug 1 : ta fonction modifie encore le tableau init qu'elle reçoit.")
        first_27 = ((Z_peng[:, None, :] - kept_27[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
        expected_27 = np.array([Z_peng[first_27 == j].mean(axis=0) for j in range(3)])   # after ONE iteration
        one_27 = np.asarray(kmeans_fixed_27(Z_peng, kept_27.copy(), max_iter=1)[0])
        verdict("7.27", one_27.shape == (3, 4) and np.allclose(one_27, expected_27),
                "bug 2 corrigé : après une itération, chaque centre est la moyenne de ses points, feature par feature.",
                "bug 2 : après une itération (max_iter=1), un centre n'est pas la moyenne de ses points, feature par feature.")
        line_27 = np.array([[0.0], [2.0], [3.0], [9.0], [11.0], [12.0]])         # the example of the course sheet (§7.5)
        final_27 = np.ravel(kmeans_fixed_27(line_27, np.array([[0.0], [3.0]]))[0])
        verdict("7.27", np.allclose(final_27, [5 / 3, 32 / 3]),
                "bug 3 corrigé : sur l'exemple de la fiche (0, 2, 3, 9, 11, 12), les centres arrivent à 5/3 et 32/3.",
                f"bug 3 : sur l'exemple de la fiche (0, 2, 3, 9, 11, 12, centres de départ 0 et 3), j'attends les centres "
                f"5/3 et 32/3, j'obtiens {final_27.round(3).tolist()} : l'algorithme s'arrête trop tôt.")
        verdict("7.27", np.isclose(inertia_27, ((Z_peng - np.asarray(centres_27)[labels_27]) ** 2).sum()),
                "bug 4 corrigé : l'inertie est la somme des CARRÉS des distances.",
                "bug 4 : l'inertie renvoyée n'est pas la somme des carrés des distances de chaque point à son centre.")
        verdict("7.27", np.allclose(centres_27, sk_27.cluster_centers_) and np.array_equal(labels_27, sk_27.labels_)
                and np.isclose(inertia_27, sk_27.inertia_),
                "ta version donne les centres, les labels et l'inertie de scikit-learn.",
                "ta version ne donne pas encore les centres, les labels et l'inertie de scikit-learn.")

### Ex 7.28 — Coefficient de silhouette 🔨 ★★★ ⏱️ 40 min
**Objectif :** programmer la silhouette, point par point puis en moyenne, et l'utiliser pour juger un découpage.  
**Prérequis :** Ex 7.13 · fiche §7.5 (encadré 🧮 sur la silhouette) · **Fil rouge :** synthétique · **mylearn :** `cluster.py` · **Parcours :** R, M, C

> **mylearn** : dans `mon_travail/mylearn/cluster.py`, mêmes règles qu'en 7.13 (NumPy permis, SciPy et scikit-learn non). Enregistre, puis relance la cellule de vérification.

Écris `silhouette_samples(X, labels)` puis `silhouette_score(X, labels)` (lis leurs docstrings et l'encadré 🧮 de la fiche) :
- valide les entrées : `ValueError` si le nombre d'étiquettes distinctes n'est pas entre 2 et $n - 1$ (la silhouette n'a pas de sens avec un seul cluster, ni avec un point par cluster) ;
- calcule la matrice des distances (la racine carrée de `pairwise_sq_distances(X, X)`) et mets sa diagonale à 0 (`np.fill_diagonal`) ;
- $a(i)$ : la distance moyenne de $i$ aux **autres** points de son cluster (divise par la taille du cluster moins 1) ; $b(i)$ : la plus petite des distances moyennes de $i$ aux points de chacun des **autres** clusters ; un masque booléen par cluster suffit ;
- $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$, avec $s(i) = 0$ pour un point seul dans son cluster, et 0 (pas `nan`) quand $a(i) = b(i) = 0$ (des points confondus) ;
- `silhouette_score` est la moyenne des $s(i)$, un `float`.

La vérification reprend l'exemple de la docstring, puis calcule la silhouette des manchots standardisés, découpés par le `KMeans` de scikit-learn :
a) avec $k = 3$ ;
b) avec $k = 2$ ;
elle compare ta silhouette à celle de scikit-learn, trace la silhouette de chaque manchot pour $k = 3$, et lance les tests.

Dans tes notes : la silhouette préfère $k = 2$, alors qu'il y a trois espèces. Pourquoi ? Lis la figure : quel cluster a les silhouettes les plus basses, et quelles espèces contient-il ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.28"):
    X_doc_28 = np.array([[0.0], [1.0], [4.0], [6.0]])
    print("docstring example:", mylearn.cluster.silhouette_samples(X_doc_28, [0, 0, 1, 1]),
          "· score", mylearn.cluster.silhouette_score(X_doc_28, [0, 0, 1, 1]))
    labels3_28 = SklearnKMeans(n_clusters=3, n_init=10, random_state=0).fit(Z_peng).labels_
    labels2_28 = SklearnKMeans(n_clusters=2, n_init=10, random_state=0).fit(Z_peng).labels_
    score3_28 = mylearn.cluster.silhouette_score(Z_peng, labels3_28)
    score2_28 = mylearn.cluster.silhouette_score(Z_peng, labels2_28)
    if returned("7.28", "silhouette_score", score3_28) and np.size(score3_28) != 1:
        print("❌ Ex 7.28 : silhouette_score doit renvoyer un seul nombre, la moyenne des silhouettes.")
    elif score3_28 is not None:
        wb.check("7.28a", score3_28, computed=True)
        wb.check("7.28b", score2_28, computed=True)
        verdict("7.28", np.isclose(float(score3_28), sklearn_silhouette_score(Z_peng, labels3_28)),
                "la même silhouette que sklearn.metrics.silhouette_score.",
                "ta silhouette diffère de celle de scikit-learn : relis a(i) et b(i).")
        print(f"silhouette score: k = 3 → {float(score3_28):.4f} · k = 2 → {float(np.ravel(score2_28)[0]):.4f}")
    values_28 = mylearn.cluster.silhouette_samples(Z_peng, labels3_28)
    if returned("7.28", "silhouette_samples", values_28) and np.shape(values_28) == (len(Z_peng),):
        fig, ax = plt.subplots(figsize=(7, 4.6))
        low_28 = 0
        for c in range(3):
            ranked = np.sort(np.asarray(values_28)[labels3_28 == c])
            ax.fill_betweenx(np.arange(low_28, low_28 + len(ranked)), 0, ranked, alpha=0.8, label=f"cluster {c}")
            low_28 += len(ranked) + 10
        ax.axvline(np.mean(values_28), color="black", ls="--", label="silhouette score (mean)")
        ax.set(xlabel="silhouette s(i)", ylabel="penguins, sorted inside each cluster", yticks=[],
               title="Silhouette of the 333 penguins, k = 3")
        ax.legend(fontsize=8)
        plt.show()
    print(pd.crosstab(labels3_28, species, rownames=["cluster (k = 3)"], colnames=["species"]))
    run_mylearn_tests("cluster", "test_silhouette_")

### Ex 7.29 — Choisir k : coude de l'inertie et silhouette, de k = 2 à 7 🔬 ★★★ ⏱️ 35 min
**Objectif :** choisir un nombre de clusters sans labels avec deux critères, et voir ce qu'ils ne peuvent pas dire.  
**Prérequis :** Ex 7.26 · Ex 7.28 · fiche §7.5 (« Combien de clusters ? ») · **Fil rouge :** synthétique · **Parcours :** R, C

Les 320 points de `X_29` n'ont pas de labels. Combien de clusters faut-il demander à k-means ?

Écris `scan_29(X, ks)`, qui renvoie deux dictionnaires `{k: valeur}`, pour chaque $k$ de `ks` : l'inertie de `mylearn.cluster.KMeans(n_clusters=k, n_init=10, random_state=0)` entraîné sur `X`, et la silhouette de ses labels (`mylearn.cluster.silhouette_score`). La vérification l'appelle pour $k$ de 2 à 7 (`K_29`) et contrôle :
a) l'inertie pour $k = 3$ ;
b) la silhouette pour $k = 3$ ;
c) la silhouette pour $k = 4$ ;
puis elle affiche le tableau et trace les deux courbes. Lis-les, puis réponds :
d) `best_k_29` : le $k$ de la plus grande silhouette (un entier) ;
e) `elbow_k_29` : le $k$ du coude, lu sur la courbe de l'inertie : le point après lequel elle ne baisse plus que lentement (un entier).

Dans tes notes : combien de paquets vois-tu sur la figure des données ? Lis la cellule qui les fabrique : combien le générateur en a-t-il utilisé ? Que disent les deux critères, et pourquoi ? Si un client te demandait quatre groupes, lui dirais-tu non ?

In [ ]:
rng_29 = np.random.default_rng(729)
BLOBS_29 = [((0.0, 0.0), 1.0), ((8.0, 0.0), 1.0), ((4.0, 7.0), 0.7), ((5.8, 7.8), 0.7)]   # (centre, standard deviation)
X_29 = np.vstack([rng_29.normal(centre, spread, (80, 2)) for centre, spread in BLOBS_29])
K_29 = range(2, 8)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_29[:, 0], X_29[:, 1], s=8, color="gray")
ax.set(title="320 points without labels", aspect="equal")
plt.show()

In [ ]:
def scan_29(X, ks):
    """({k: inertia_}, {k: silhouette score}) of mylearn's KMeans(n_clusters=k, n_init=10, random_state=0) on X."""
    raise NotImplementedError("scan_29() is not written yet")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.29"):
    scan_result_29 = scan_29(X_29, K_29)
    if returned("7.29", "scan_29", scan_result_29) and not (
            isinstance(scan_result_29, (tuple, list)) and len(scan_result_29) == 2
            and all(isinstance(part, dict) for part in scan_result_29)):
        print("❌ Ex 7.29 : scan_29 doit renvoyer deux dictionnaires {k: valeur} : (inerties, silhouettes).")
    elif scan_result_29 is not None:
        inertias_29, silhouettes_29 = scan_result_29
        wb.check("7.29a", inertias_29[3], computed=True)
        wb.check("7.29b", silhouettes_29[3], computed=True)
        wb.check("7.29c", silhouettes_29[4], computed=True)
        print(pd.DataFrame({"inertia": inertias_29, "silhouette": silhouettes_29}).round(4))
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
        axes[0].plot(list(K_29), [inertias_29[k] for k in K_29], "o-")
        axes[0].set(xlabel="k", ylabel="inertia", title="The elbow method")
        axes[1].plot(list(K_29), [silhouettes_29[k] for k in K_29], "o-", color="C1")
        axes[1].set(xlabel="k", ylabel="silhouette score", title="The silhouette")
        plt.show()

Lis le tableau et les deux courbes, puis réponds à d) et e).

In [ ]:
best_k_29 = ...    # d) the k with the highest silhouette (an integer)
elbow_k_29 = ...   # e) the k of the elbow, read on the curve of the inertia (an integer)

In [ ]:
wb.check("7.29d", best_k_29)
wb.check("7.29e", elbow_k_29)

### Ex 7.30 — Phénomène de Hughes : des features de bruit qui font chuter l'accuracy 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer l'effet de features sans information sur deux classifieurs, et relier la baisse à la concentration des distances.  
**Prérequis :** Ex 7.14 · Ex 7.20 · ch. 1 (jeux d'entraînement et de test) · fiche §7.6 · **Fil rouge :** Penguins · **Parcours :** C

On classe les manchots (233 pour l'entraînement, 100 pour le test, le découpage du ch. 1) avec des features de plus en plus nombreuses : d'abord 1 à 4 vraies mesures (standardisées avec les manchots d'entraînement, dans l'ordre de `FEATURES`), puis les 4 mesures suivies de 2 à 1 000 features de **bruit pur**, tirées d'une loi normale, sans aucun lien avec l'espèce. `features_30(n_real, n_noise)` (fourni) renvoie ce tableau pour les 333 manchots. Deux classifieurs : ton `NearestCentroid` et le **1-plus-proche-voisin** (1-NN), qui donne à chaque point le label du point d'entraînement le plus proche.

Écris deux fonctions :
- `nn1_predict_30(X_train, y_train, X_test)` : le 1-NN, avec ta `pairwise_sq_distances` (7.13) et `argmin` ;
- `hughes_30(n_real, n_noise)`, qui renvoie la liste `[accuracy du centroïde le plus proche, accuracy du 1-NN]`, entraînés sur les lignes `TRAIN_PENG` de `features_30(n_real, n_noise)` et évalués sur les lignes `TEST_PENG`.

La vérification calcule la courbe pour toutes les configurations de `CONFIGS_30`, puis vérifie les deux accuracies dans quatre d'entre elles :
a) 2 vraies mesures ;
b) les 4 vraies mesures ;
c) les 4 mesures et 100 features de bruit ;
d) les 4 mesures et 1 000 features de bruit ;
et trace la courbe.

Dans tes notes : où est le sommet de la courbe ? Pourquoi le 1-NN s'effondre-t-il bien plus vite que le centroïde le plus proche ? (pense à la moyenne des features de bruit sur les dizaines de manchots d'une classe, puis à 7.20) « Plus de features, c'est toujours mieux » : vrai ou faux ?

In [ ]:
mean_30, std_30 = X_peng[TRAIN_PENG].mean(axis=0), X_peng[TRAIN_PENG].std(axis=0)
Z_30 = (X_peng - mean_30) / std_30                             # standardised with the training penguins only
NOISE_30 = np.random.default_rng(730).normal(size=(len(X_peng), 1000))   # 1000 features of pure noise
CONFIGS_30 = [(1, 0), (2, 0), (3, 0), (4, 0), (4, 2), (4, 5), (4, 10), (4, 20), (4, 50), (4, 100), (4, 200), (4, 500),
              (4, 1000)]                                       # (number of real features, number of noise features)


def features_30(n_real, n_noise):
    """The first n_real measurements (standardised), then n_noise features of noise, for the 333 penguins."""
    return np.hstack([Z_30[:, :n_real], NOISE_30[:, :n_noise]])

In [ ]:
def nn1_predict_30(X_train, y_train, X_test):
    """1-nearest neighbour: for every row of X_test, the label of the closest row of X_train."""
    raise NotImplementedError("nn1_predict_30() is not written yet")


def hughes_30(n_real, n_noise):
    """[accuracy of NearestCentroid, accuracy of the 1-NN] on TEST_PENG, trained on TRAIN_PENG of features_30(...)."""
    raise NotImplementedError("hughes_30() is not written yet")

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.30"):
    curve_30 = {config: hughes_30(*config) for config in CONFIGS_30}
    if returned("7.30", "hughes_30", curve_30[(1, 0)]) and np.size(curve_30[(1, 0)]) != 2:
        print("❌ Ex 7.30 : hughes_30 doit renvoyer deux accuracies : [centroïde le plus proche, 1-NN].")
    elif curve_30[(1, 0)] is not None:
        curve_30 = {config: [float(v) for v in np.ravel(pair)] for config, pair in curve_30.items()}
        wb.check("7.30a", curve_30[(2, 0)], computed=True)
        wb.check("7.30b", curve_30[(4, 0)], computed=True)
        wb.check("7.30c", curve_30[(4, 100)], computed=True)
        wb.check("7.30d", curve_30[(4, 1000)], computed=True)
        print(pd.DataFrame([[r, m, r + m, *curve_30[(r, m)]] for r, m in CONFIGS_30],
                           columns=["real", "noise", "features", "nearest centroid", "1-NN"]).to_string(index=False))
        total_30 = [r + m for r, m in CONFIGS_30]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(total_30, [curve_30[c][0] for c in CONFIGS_30], "o-", label="nearest centroid")
        ax.plot(total_30, [curve_30[c][1] for c in CONFIGS_30], "s-", label="1-nearest neighbour")
        ax.axvline(4, color="gray", ls=":", label="the 4 real measurements")
        ax.set(xscale="log", xlabel="number of features (real, then noise)", ylabel="accuracy on the 100 test penguins",
               ylim=(0.3, 1.02), title="Hughes phenomenon: features that bring nothing")
        ax.legend(fontsize=8)
        plt.show()

### Ex 7.31 — Défi : retrouver les espèces de manchots sans labels 🏆 ★★★ ⏱️ 60 min
**Objectif :** trouver une représentation des données qui fait apparaître les espèces dans les clusters de k-means, et qui tient sur des données qu'elle n'a pas vues.  
**Prérequis :** Ex 7.17 · Ex 7.26 · ch. 2 (corrélation) · fiche §7.5 et §7.6 · **Fil rouge :** Penguins · **Parcours :** C

En 7.17 c), k-means sur les quatre mesures standardisées n'atteint pas une pureté de 0,95. Fais mieux, **sans** les espèces. Les règles :
- tu écris `cluster_31(X)`, qui reçoit les **4 mesures brutes** de manchots (un tableau de forme (n, 4), colonnes dans l'ordre de `FEATURES`) et renvoie un label de cluster par manchot ;
- le clustering lui-même est fait par **ton** `mylearn.cluster.KMeans`, avec `n_clusters=3` ; avant, tu peux choisir, combiner, transformer ou pondérer les mesures comme tu veux ;
- tes choix ne regardent jamais les espèces (ni le sexe, ni l'île) : pas de seuil réglé à la main sur les labels. Ta fonction recalcule ses statistiques (moyennes, écarts-types…) sur le `X` qu'elle reçoit ;
- `grade_31` (fourni) lance ta fonction sur les 333 manchots, puis sur les manchots de chaque année **séparément** (2007, 2008, 2009 : une centaine chacune), et calcule la pureté (7.17) par rapport aux espèces.

**Objectif : une pureté d'au moins 0,95 sur les 333 manchots, et d'au moins 0,90 pour chaque année.** Le point de départ fourni, celui de 7.17 avec ton `KMeans`, n'y arrive pas.

Dans tes notes : ton idée, pourquoi elle marche, et ce qu'elle suppose. Une piste : la matrice de corrélation des quatre mesures (ch. 2), et ce qui distingue un Adélie d'un Chinstrap.

In [ ]:
def purity(labels, truth):
    """Purity of a clustering (Ex 7.17): each cluster takes its most frequent true class."""
    table = pd.crosstab(np.asarray(labels), np.asarray(truth))
    return float(table.max(axis=1).sum() / len(labels))


def grade_31(cluster):
    """(purities, "ok"): cluster(X) on the 333 penguins, then on each year clustered separately (X: the 4 raw
    measurements); (None, the reason) when the labels returned cannot be graded."""
    results = {}
    subsets = [("all years", np.arange(len(X_peng)))] + [(str(year), np.flatnonzero(years == year))
                                                           for year in (2007, 2008, 2009)]
    for name, rows in subsets:
        labels = cluster(X_peng[rows].copy())
        if labels is None:
            return None, "cluster_31 renvoie None : as-tu oublié le return ?"
        labels = np.asarray(labels)
        if labels.shape != (len(rows),):
            return None, (f"cluster_31 doit renvoyer un label par manchot : {len(rows)} labels attendus ({name}), "
                          f"j'obtiens un tableau de forme {labels.shape}.")
        if len(np.unique(labels)) != 3:
            return None, (f"cluster_31 doit renvoyer exactement 3 clusters ({name}) : j'en compte "
                          f"{len(np.unique(labels))}.")
        results[name] = purity(labels, species[rows])
    return results, "ok"

In [ ]:
def cluster_31(X):
    """3 clusters of the penguins described by X (their 4 raw measurements), found by mylearn's KMeans."""
    Z = (X - X.mean(axis=0)) / X.std(axis=0)                  # the starting point: Ex 7.17 with YOUR KMeans
    model = mylearn.cluster.KMeans(n_clusters=3, n_init=10, random_state=0)
    model.fit(Z)
    return model.labels_

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="7")   # reload your saved file
with wb.attempt("7.31"):
    scores_31, message_31 = grade_31(cluster_31)
    if scores_31 is None:
        print("❌ Ex 7.31 :", message_31)
    else:
        print({name: round(value, 3) for name, value in scores_31.items()})
        verdict("7.31", scores_31["all years"] >= 0.95,
                f"pureté {fr(scores_31['all years'])} sur les 333 manchots : objectif 0,95 atteint.",
                f"pureté {fr(scores_31['all years'])} sur les 333 manchots : l'objectif est 0,95.")
        worst_31 = min(("2007", "2008", "2009"), key=scores_31.get)
        verdict("7.31", scores_31[worst_31] >= 0.90,
                f"pureté d'au moins 0,90 pour chaque année traitée seule (la plus basse : {worst_31}, "
                f"{fr(scores_31[worst_31])}).",
                f"pureté {fr(scores_31[worst_31])} pour les manchots de {worst_31}, traités seuls : l'objectif est "
                f"0,90 pour chaque année.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu choisir un seuil de décision d'après le coût des deux erreurs, et dire qui doit fixer ces coûts ?
2. Sais-tu compter et dépouiller les votes d'un un-contre-un, et dire quand le préférer à l'un-contre-tous ?
3. Sais-tu dérouler k-means à la main, et expliquer pourquoi la densité des exemples s'effondre quand on ajoute des features ?

**Pour aller plus loin** : les guides « Clustering » et « Multiclass and multioutput algorithms » de scikit-learn, cités dans la fiche. La suite : le ch. 8 (entraînement et test), où un jeu de validation fixe les seuils et les hyperparamètres comme $k$.